# 🌌 NucleiSky3D App
---

Welcome to **NucleiSky3D**, a constellation-based registration framework designed to help you easily align **3D microscopy volumes**. By matching the spatial arrangement of segmented cell nuclei, this notebook estimates a 3D **similarity transform** (3D rotation, uniform scale, and translation) between your **reference volume** (the “sky map”) and a **moving crop** (the “telescope snapshot”).

Instead of relying on tricky pixel-intensity correlations, NucleiSky treats your nuclei as a constellation of biological landmarks. It searches for geometrically consistent patterns between these constellations, making your alignments incredibly robust to partial overlaps, missing nuclei, segmentation noise, and differing magnifications.

## Core Capabilities

* **Constellation Matching:** Align your volumes based on the true 3D spatial geometry of your nuclei, entirely independent of fluctuating pixel intensities.
* **Scale & Rotation Independence:** Because matching happens in real physical units (µm), you can seamlessly align images taken at different magnifications or at arbitrary 3D orientations.
* **High Reliability:** Built-in outlier filtering and consensus scoring automatically ignore noise and find the true, coherent subset of nuclei that supports your alignment.
* **Modular Design:** Run the entire process end-to-end right here in the notebook, or effortlessly swap in your own external segmentation labels.

## Your Matching Pipeline

### Step 1: Segmentation

This notebook gives you flexible options for detecting your nuclei:
* **Built-in 2.5D Segmentation:** A fast, memory-efficient, and highly effective built-in tool! We segment your image slice-by-slice and seamlessly stitch the results into a cohesive 3D volume using optimized overlap tracking. It is a fantastic way to get great results right out of the box.
* **Bring Your Own Labels:** Already have a favorite 3D segmentation tool? You can easily plug in your own labeled volumes (e.g., from Cellpose3D, StarDist3D, or Ilastik) for maximum control over your pipeline.

### Step 2 - Option A: 3D Matching Strategies

Under the hood, NucleiSky3D uses two powerful matching engines to find your alignment:
* **Pyramid Matcher:** Our default engine. It excels at handling smaller or sparser point clouds (`< 1000` nuclei) by building local geometric graphs to find exact matches.
* **Geometric Hashing:** Designed for large, dense constellations (`>= 1000` nuclei), using a highly effective invariant voting scheme to power through massive datasets.

### Step 2 - Option B: Adaptive "Auto-Pilot" Mode

Not sure which matcher or settings to use? Let the **Adaptive Matcher** do the heavy lifting. It acts as an auto-pilot, analyzing the size of your dataset to automatically choose the best matching strategy and dynamically adjust thresholds. It will find the first high-confidence solution and automatically export your results.

### Step 3: Apply & Export

Once you compute a successful transform using your nuclei channel, you can easily **save it and re-apply it** to other channels (like actin, membranes, or specific proteins) within the same crop. By the end of this notebook, you will be able to export beautifully aligned 3D TIFFs ready for your downstream analysis.

# Step 0: Initialize Environment & Install Dependencies
---


**Run the following cell to initialize the environment.**

* **For Google Colab:** The notebook will automatically detect the environment, mount Google Drive, and install the necessary libraries.

* **For Local Jupyter Lab:** The notebook will skip installation steps to avoid messing up your local environment.



In [ ]:
# @title  Initialize Environment & Install Dependencies

from pathlib import Path
import sys

# Metadata
current_version = "0.0.2"
notebook_name = "NucleiSky3DApp"

# ---------------------------------------------------------
# 1. Environment Detection & Installation (Colab Only)
# ---------------------------------------------------------
if 'google.colab' in sys.modules:
    print("🚀 Detected Google Colab. Starting installation...")

    from google.colab import userdata

    # 1. Retrieve the token securely from Colab Secrets
    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        print("Error: Secret 'GITHUB_TOKEN' not found. Please add it in the sidebar (🔑).")
        token = None

    # 2. Install from your private repo
    if token:
        user = 'CellMigrationLab'  # Replace with your GitHub username
        repo = 'NucleiSky-test'

        # Incompatibilities due to the newest scikit-image==0.26.0 version
        !pip install -q cucim-cu12==26.4.0
        # We use the PEP 508 syntax: "package[extras] @ url"
        !pip install -q "nucleisky[all] @ git+https://{token}@github.com/{user}/{repo}.git"

    from google.colab import drive
    drive.mount('/content/gdrive')

    print("✅ Colab setup done")

else:
    # Fallback for local environments
    print("done")

# ---------------------------------------------------------
# 2. GPU Check
# ---------------------------------------------------------
try:
    import torch
    if torch.cuda.is_available():
            print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠️  WARNING: No GPU detected. Segmentation with Cellpose or InstanSeg will be VERY SLOW on CPU.")
        if 'google.colab' in sys.modules:
            print("Go to Runtime > Change runtime type > Hardware accelerator > T4 GPU")
except ImportError:
    print("⚠️  WARNING: PyTorch not installed. Cellpose and InstanSeg segmentation will not work. If you want to use these features, please check the installation instructions.")

# ---------------------------------------------------------
# 3. Main Imports
# ---------------------------------------------------------
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import ipywidgets as widgets
from pathlib import Path
import numpy as np
import traceback
import tifffile
import html
import json
import copy

from nucleisky3d.export import export_aligned_crop_tiff, export_bbox_pair_tiffs_3d, run_export_with_record_3d
from nucleisky3d.matching.geometry import bbox_full_px_from_similarity_um_3d, bbox_add_margin_px_3d
from nucleisky3d.preprocess import ij_percentile_normalize, scale_normalize_pair_for_segmentation
from nucleisky3d.config import DEFAULT_MATCHER_CONFIG, save_matcher_config_json
from nucleisky3d.visualization import imshow_safe3d, plot_warp_overlay3d
from nucleisky3d.demo_utils import generate_random_subvolume_3d
from nucleisky3d.features import extract_nuclear_features_3d
from nucleisky3d.segmentation import segment_nuclei_2p5d

from nucleisky3d.io import (
    inspect_volume_header, 
    load_volume, make_result_dir, 
    require_voxel_size_um_zyx, 
    save_tiff_zyx, 
    save_nucleisky_transform_3d, 
    append_transform_jsonl,
    load_transforms_any_3d
)

from nucleisky3d.pipeline import (
    NucleiSky3D,
    centroids_from_df_3d, 
    pick_best_transform_3d, 
    run_adaptive_matching_and_export_3d
)



# ----------------------------
# UI Styling & CSS
# ----------------------------

STYLE_CSS = """
<style>
:root {
    --ns-bg:              #FFFFFF;
    --ns-bg-page:         #F1F5F9;
    --ns-bg-soft:         #F8FAFC;
    --ns-bg-softer:       #F9FAFB;
    --ns-border:          #CBD5E1;
    --ns-border-soft:     #E2E8F0;
    --ns-text:            #1E293B;
    --ns-text-muted:      #334155;
    --ns-text-soft:       #475569;
    --ns-label:           #475569;

    --ns-ok-bg:           #ECFDF5;
    --ns-ok-border:       #6EE7B7;
    --ns-ok-text:         #065F46;

    --ns-err-bg:          #FEF2F2;
    --ns-err-border:      #FCA5A5;
    --ns-err-text:        #991B1B;

    --ns-warn-bg:         #FFFBEB;
    --ns-warn-border:     #FCD34D;
    --ns-warn-text:       #92400E;

    --ns-accent:          #2563EB;
    --ns-accent-hover:    #1D4ED8;
    --ns-accent-soft:     #DBEAFE;

    --ns-info-bg:         #0284C7;
    --ns-info-hover:      #0369A1;
    --ns-info-text:       #FFFFFF;

    --ns-spinner-track:   #BFDBFE;

    --ns-btn-bg:          #F1F5F9;
    --ns-btn-hover-bg:    #E2E8F0;
    --ns-btn-text:        #334155;
    --ns-btn-border:      #CBD5E1;
    --ns-btn-active-bg:   #2563EB;
    --ns-btn-active-text: #FFFFFF;

    --ns-disabled-bg:     #EFF6FF;
    --ns-disabled-text:   #1E40AF;
    --ns-disabled-border: #BFDBFE;

    --ns-disabled-strong-bg:     #DBEAFE;
    --ns-disabled-strong-text:   #1E3A8A;
    --ns-disabled-strong-border: #93C5FD;
}

/* ---------------------------------------------------------
   Base layout
--------------------------------------------------------- */

.ns-root {
    background-color: var(--ns-bg-page) !important;
    padding: 20px;
    border-radius: 10px;
    color: var(--ns-text) !important;
    color-scheme: light !important;
}

.ns-card {
    border: 1px solid var(--ns-border);
    border-radius: 8px;
    padding: 16px;
    margin-bottom: 16px;
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    box-shadow: 0 1px 3px rgba(0,0,0,0.06);
    color-scheme: light !important;
}

.ns-card-warn {
    border: 1px solid var(--ns-err-border);
    background-color: var(--ns-err-bg) !important;
}

.ns-header {
    font-weight: 700;
    font-size: 13px;
    color: var(--ns-text-soft) !important;
    margin-bottom: 12px;
    text-transform: uppercase;
    letter-spacing: 0.06em;
    border-bottom: 1px solid var(--ns-border-soft);
    padding-bottom: 8px;
}

.ns-label {
    font-size: 11px;
    font-weight: 700;
    color: var(--ns-label) !important;
    margin-top: 8px;
    margin-bottom: 4px;
    text-transform: uppercase;
    letter-spacing: 0.04em;
}

.ns-page-title {
    font-weight: 800;
    font-size: 18px;
    color: var(--ns-text) !important;
    margin: 0 0 4px 0;
}

.ns-page-subtitle {
    font-size: 13px;
    color: var(--ns-text-muted) !important;
    margin-bottom: 16px;
}

.widget-label {
    color: var(--ns-text-muted) !important;
    font-weight: 500 !important;
}

.widget-label-basic {
    color: var(--ns-text-muted) !important;
}

.ns-subhead {
    font-size: 11px;
    font-weight: 700;
    color: var(--ns-label) !important;
    text-transform: uppercase;
    letter-spacing: 0.06em;
    margin: 8px 0 6px 0;
}

.ns-body-text {
    font-size: 12px;
    color: var(--ns-text-soft) !important;
    line-height: 1.5;
}

.ns-body-text b,
.ns-body-text strong {
    color: var(--ns-text) !important;
}

.ns-body-text ul {
    margin: 6px 0 0 18px;
    padding: 0;
}

.ns-body-text li {
    margin: 2px 0;
}

.ns-desc {
    font-size: 12px;
    color: var(--ns-text-soft) !important;
    margin-bottom: 8px;
    font-style: italic;
}

/* ---------------------------------------------------------
   Status messages
--------------------------------------------------------- */

.ns-status {
    font-size: 13px;
    font-weight: 600;
    margin: 8px 0;
}

.ns-status-running {
    display: flex;
    align-items: center;
    gap: 10px;
    color: var(--ns-accent) !important;
}

.ns-status-ok {
    color: var(--ns-ok-text) !important;
}

.ns-status-warn {
    color: var(--ns-warn-text) !important;
}

.ns-status-error {
    color: var(--ns-err-text) !important;
}

.ns-message {
    width: 100% !important;
    box-sizing: border-box !important;
    border-radius: 8px !important;
    padding: 14px 16px !important;
    margin: 12px 0 !important;
    line-height: 1.45 !important;
    color-scheme: light !important;
}

.ns-message-title {
    font-weight: 800 !important;
    font-size: 14px !important;
    margin-bottom: 6px !important;
}

.ns-message-body {
    font-size: 13px !important;
    line-height: 1.45 !important;
}

.ns-message-error {
    border: 1px solid var(--ns-err-border) !important;
    background-color: var(--ns-err-bg) !important;
    color: var(--ns-err-text) !important;
}

.ns-message-error .ns-message-title,
.ns-message-error .ns-message-body {
    color: var(--ns-err-text) !important;
}

.ns-message-ok {
    border: 1px solid var(--ns-ok-border) !important;
    background-color: var(--ns-ok-bg) !important;
    color: var(--ns-ok-text) !important;
}

.ns-message-ok .ns-message-title,
.ns-message-ok .ns-message-body {
    color: var(--ns-ok-text) !important;
}

.ns-message-warn {
    border: 1px solid var(--ns-warn-border) !important;
    background-color: var(--ns-warn-bg) !important;
    color: var(--ns-warn-text) !important;
}

.ns-message-warn .ns-message-title,
.ns-message-warn .ns-message-body {
    color: var(--ns-warn-text) !important;
}

.ns-spinner {
    width: 14px;
    height: 14px;
    border: 2px solid var(--ns-spinner-track);
    border-top: 2px solid var(--ns-accent);
    border-radius: 50%;
    animation: ns-spin 0.8s linear infinite;
}

.ns-step-active {
    background: var(--ns-spinner-track) !important;
    color: var(--ns-accent) !important;
}

.ns-step-idle {
    background: var(--ns-bg-soft) !important;
    color: var(--ns-text-soft) !important;
}

/* ---------------------------------------------------------
   Toggle buttons
--------------------------------------------------------- */

.widget-toggle-buttons .widget-toggle-button {
    background: var(--ns-btn-bg) !important;
    color: var(--ns-btn-text) !important;
    border: 1px solid var(--ns-btn-border) !important;
}

.widget-toggle-buttons .widget-toggle-button.mod-active {
    background: var(--ns-btn-active-bg) !important;
    color: var(--ns-btn-active-text) !important;
    border-color: var(--ns-btn-active-bg) !important;
}

/* ---------------------------------------------------------
   General widget light-theme forcing inside cards
--------------------------------------------------------- */

.ns-card .widget-box,
.ns-card .widget-hbox,
.ns-card .widget-vbox,
.ns-card .widget-gridbox,
.ns-card .widget-label,
.ns-card .widget-label-basic,
.ns-card .widget-html,
.ns-card .widget-html-content,
.ns-card .widget-output,
.ns-card .jupyter-widgets,
.ns-card .p-Widget,
.ns-card .lm-Widget {
    background-color: transparent !important;
    color: var(--ns-text) !important;
    color-scheme: light !important;
}

.ns-card select,
.ns-card input,
.ns-card textarea,
.ns-card button {
    color-scheme: light !important;
}

.ns-card select,
.ns-card input,
.ns-card textarea {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    border-color: var(--ns-border) !important;
}

.ns-card select:disabled,
.ns-card input:disabled,
.ns-card textarea:disabled {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text-soft) !important;
    border-color: var(--ns-border-soft) !important;
    opacity: 1 !important;
}

/* ---------------------------------------------------------
   Button hard fix
--------------------------------------------------------- */

.ns-card .widget-button,
.ns-root .widget-button,
.ns-tabs .widget-button,
.ns-tab-panel .widget-button,
.ns-accordion .widget-button,
.ns-accordion-body .widget-button,
button.widget-button {
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    gap: 6px !important;

    box-sizing: border-box !important;
    line-height: 1 !important;
    text-align: center !important;
    vertical-align: middle !important;

    background-color: var(--ns-btn-bg) !important;
    color: var(--ns-btn-text) !important;
    border: 1px solid var(--ns-btn-border) !important;
    border-radius: 6px !important;
    font-weight: 700 !important;
    box-shadow: 0 1px 2px rgba(15,23,42,0.10) !important;

    min-height: 32px !important;
    padding: 0 12px !important;
    cursor: pointer !important;
    opacity: 1 !important;
    color-scheme: light !important;
}

.ns-card .widget-button i,
.ns-root .widget-button i,
.ns-tabs .widget-button i,
.ns-tab-panel .widget-button i,
.ns-accordion .widget-button i,
.ns-accordion-body .widget-button i,
button.widget-button i,
.ns-card .widget-button .fa,
.ns-root .widget-button .fa,
.ns-tabs .widget-button .fa,
.ns-tab-panel .widget-button .fa,
.ns-accordion .widget-button .fa,
.ns-accordion-body .widget-button .fa,
button.widget-button .fa {
    line-height: 1 !important;
    margin: 0 !important;
    padding: 0 !important;
    display: inline-flex !important;
    align-items: center !important;
}

.ns-card .widget-button:hover,
.ns-root .widget-button:hover,
.ns-tabs .widget-button:hover,
.ns-tab-panel .widget-button:hover,
.ns-accordion .widget-button:hover,
.ns-accordion-body .widget-button:hover,
button.widget-button:hover {
    background-color: var(--ns-btn-hover-bg) !important;
    color: var(--ns-text) !important;
    border-color: var(--ns-border) !important;
}

.ns-card .widget-button.mod-info,
.ns-root .widget-button.mod-info,
.ns-tabs .widget-button.mod-info,
.ns-tab-panel .widget-button.mod-info,
.ns-accordion .widget-button.mod-info,
.ns-accordion-body .widget-button.mod-info,
button.widget-button.mod-info,
.ns-action-button {
    background-color: var(--ns-info-bg) !important;
    color: var(--ns-info-text) !important;
    border-color: var(--ns-info-bg) !important;
    box-shadow: 0 2px 4px rgba(2,132,199,0.22) !important;
}

.ns-card .widget-button.mod-info:hover,
.ns-root .widget-button.mod-info:hover,
.ns-tabs .widget-button.mod-info:hover,
.ns-tab-panel .widget-button.mod-info:hover,
.ns-accordion .widget-button.mod-info:hover,
.ns-accordion-body .widget-button.mod-info:hover,
button.widget-button.mod-info:hover,
.ns-action-button:hover {
    background-color: var(--ns-info-hover) !important;
    color: var(--ns-info-text) !important;
    border-color: var(--ns-info-hover) !important;
}

.ns-card .widget-button.mod-primary,
.ns-root .widget-button.mod-primary,
.ns-tabs .widget-button.mod-primary,
.ns-tab-panel .widget-button.mod-primary,
.ns-accordion .widget-button.mod-primary,
.ns-accordion-body .widget-button.mod-primary,
button.widget-button.mod-primary,
.ns-primary-button {
    background-color: var(--ns-accent) !important;
    color: #FFFFFF !important;
    border-color: var(--ns-accent) !important;
    box-shadow: 0 2px 4px rgba(37,99,235,0.22) !important;
}

.ns-card .widget-button.mod-primary:hover,
.ns-root .widget-button.mod-primary:hover,
.ns-tabs .widget-button.mod-primary:hover,
.ns-tab-panel .widget-button.mod-primary:hover,
.ns-accordion .widget-button.mod-primary:hover,
.ns-accordion-body .widget-button.mod-primary:hover,
button.widget-button.mod-primary:hover,
.ns-primary-button:hover {
    background-color: var(--ns-accent-hover) !important;
    color: #FFFFFF !important;
    border-color: var(--ns-accent-hover) !important;
}

.ns-card .widget-button.mod-success,
.ns-root .widget-button.mod-success,
button.widget-button.mod-success {
    background-color: var(--ns-ok-text) !important;
    color: #FFFFFF !important;
    border-color: var(--ns-ok-text) !important;
}

.ns-card .widget-button.mod-warning,
.ns-root .widget-button.mod-warning,
button.widget-button.mod-warning {
    background-color: #D97706 !important;
    color: #FFFFFF !important;
    border-color: #D97706 !important;
}

.ns-card .widget-button.mod-danger,
.ns-root .widget-button.mod-danger,
button.widget-button.mod-danger {
    background-color: var(--ns-err-text) !important;
    color: #FFFFFF !important;
    border-color: var(--ns-err-text) !important;
}

/* Disabled buttons: visible but clearly disabled */
.ns-card .widget-button:disabled,
.ns-root .widget-button:disabled,
.ns-tabs .widget-button:disabled,
.ns-tab-panel .widget-button:disabled,
.ns-accordion .widget-button:disabled,
.ns-accordion-body .widget-button:disabled,
button.widget-button:disabled {
    background-color: var(--ns-disabled-bg) !important;
    color: var(--ns-disabled-text) !important;
    border-color: var(--ns-disabled-border) !important;
    box-shadow: none !important;
    cursor: not-allowed !important;
    opacity: 1 !important;
}

/* Disabled primary buttons should remain recognisable */
.ns-card .widget-button.mod-primary:disabled,
.ns-root .widget-button.mod-primary:disabled,
.ns-tabs .widget-button.mod-primary:disabled,
.ns-tab-panel .widget-button.mod-primary:disabled,
.ns-accordion .widget-button.mod-primary:disabled,
.ns-accordion-body .widget-button.mod-primary:disabled,
button.widget-button.mod-primary:disabled,
.ns-primary-button:disabled {
    background-color: var(--ns-disabled-strong-bg) !important;
    color: var(--ns-disabled-strong-text) !important;
    border-color: var(--ns-disabled-strong-border) !important;
    font-weight: 800 !important;
    opacity: 1 !important;
}

/* ---------------------------------------------------------
   Accordion hard fix for Colab dark theme
--------------------------------------------------------- */

.ns-accordion,
.ns-accordion-body {
    color-scheme: light !important;
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
}

.widget-accordion.ns-accordion,
.ns-accordion.widget-accordion,
.ns-card .widget-accordion,
.widget-accordion.ns-card {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    border: 1px solid var(--ns-border-soft) !important;
    border-radius: 8px !important;
    overflow: hidden !important;
    color-scheme: light !important;
}

.ns-accordion .p-AccordionPanel,
.ns-accordion .lm-AccordionPanel,
.ns-card .p-AccordionPanel,
.ns-card .lm-AccordionPanel {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    border: none !important;
    color-scheme: light !important;
}

.ns-accordion .p-AccordionPanel-title,
.ns-accordion .lm-AccordionPanel-title,
.ns-card .p-AccordionPanel-title,
.ns-card .lm-AccordionPanel-title {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text) !important;
    border-bottom: 1px solid var(--ns-border-soft) !important;
    font-weight: 700 !important;
    min-height: 32px !important;
    padding: 6px 10px !important;
    color-scheme: light !important;
}

.ns-accordion .p-AccordionPanel-title:hover,
.ns-accordion .lm-AccordionPanel-title:hover,
.ns-accordion .p-AccordionPanel-title.p-mod-current,
.ns-accordion .lm-AccordionPanel-title.lm-mod-current,
.ns-card .p-AccordionPanel-title:hover,
.ns-card .lm-AccordionPanel-title:hover,
.ns-card .p-AccordionPanel-title.p-mod-current,
.ns-card .lm-AccordionPanel-title.lm-mod-current {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text) !important;
}

.ns-accordion .p-AccordionPanel-titleLabel,
.ns-accordion .lm-AccordionPanel-titleLabel,
.ns-card .p-AccordionPanel-titleLabel,
.ns-card .lm-AccordionPanel-titleLabel {
    color: var(--ns-text) !important;
}

.ns-accordion .p-AccordionPanel-titleIcon,
.ns-accordion .lm-AccordionPanel-titleIcon,
.ns-card .p-AccordionPanel-titleIcon,
.ns-card .lm-AccordionPanel-titleIcon {
    color: var(--ns-text-soft) !important;
}

.ns-accordion .p-AccordionPanel-widget,
.ns-accordion .lm-AccordionPanel-widget,
.ns-accordion .p-AccordionPanel-content,
.ns-accordion .lm-AccordionPanel-content,
.ns-card .p-AccordionPanel-widget,
.ns-card .lm-AccordionPanel-widget,
.ns-card .p-AccordionPanel-content,
.ns-card .lm-AccordionPanel-content {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    padding: 12px !important;
    color-scheme: light !important;
}

.ns-accordion-body,
.widget-box.ns-accordion-body,
.widget-vbox.ns-accordion-body,
.widget-hbox.ns-accordion-body,
.ns-accordion-body.widget-box,
.ns-accordion-body.widget-vbox,
.ns-accordion-body.widget-hbox,
.ns-accordion-body.p-Widget,
.ns-accordion-body.lm-Widget {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    color-scheme: light !important;
}

.ns-accordion-body .widget-box,
.ns-accordion-body .widget-vbox,
.ns-accordion-body .widget-hbox,
.ns-accordion-body .widget-gridbox,
.ns-accordion-body .widget-label,
.ns-accordion-body .widget-label-basic,
.ns-accordion-body .widget-html,
.ns-accordion-body .widget-html-content,
.ns-accordion-body .widget-output,
.ns-accordion-body .jupyter-widgets,
.ns-accordion-body .p-Widget,
.ns-accordion-body .lm-Widget {
    background-color: transparent !important;
    color: var(--ns-text) !important;
    color-scheme: light !important;
}

.ns-accordion-body select,
.ns-accordion-body input,
.ns-accordion-body textarea,
.ns-accordion-body button,
.ns-accordion select,
.ns-accordion input,
.ns-accordion textarea,
.ns-accordion button {
    color-scheme: light !important;
}

.ns-accordion-body select,
.ns-accordion-body input,
.ns-accordion-body textarea,
.ns-accordion select,
.ns-accordion input,
.ns-accordion textarea {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    border-color: var(--ns-border) !important;
}

.ns-accordion-body select:disabled,
.ns-accordion-body input:disabled,
.ns-accordion-body textarea:disabled,
.ns-accordion select:disabled,
.ns-accordion input:disabled,
.ns-accordion textarea:disabled {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text-soft) !important;
    border-color: var(--ns-border-soft) !important;
    opacity: 1 !important;
}

/* ---------------------------------------------------------
   Tab hard fix for Colab dark theme
--------------------------------------------------------- */

.ns-tabs,
.ns-tab-panel {
    color-scheme: light !important;
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
}

.widget-tab.ns-tabs,
.ns-tabs.widget-tab,
.ns-card .widget-tab {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    border: 1px solid var(--ns-border-soft) !important;
    border-radius: 8px !important;
    overflow: hidden !important;
    color-scheme: light !important;
}

.ns-tabs .p-TabPanel,
.ns-tabs .lm-TabPanel,
.ns-card .p-TabPanel,
.ns-card .lm-TabPanel {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    color-scheme: light !important;
}

.ns-tabs .p-TabBar,
.ns-tabs .lm-TabBar,
.ns-tabs .p-TabBar-content,
.ns-tabs .lm-TabBar-content,
.ns-card .p-TabBar,
.ns-card .lm-TabBar,
.ns-card .p-TabBar-content,
.ns-card .lm-TabBar-content {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text) !important;
    border-bottom: 1px solid var(--ns-border-soft) !important;
    color-scheme: light !important;
}

.ns-tabs .p-TabBar-tab,
.ns-tabs .lm-TabBar-tab,
.ns-card .p-TabBar-tab,
.ns-card .lm-TabBar-tab {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text-soft) !important;
    border: none !important;
    border-right: 1px solid var(--ns-border-soft) !important;
    padding: 8px 12px !important;
    min-height: 34px !important;
    font-weight: 700 !important;
    color-scheme: light !important;
}

.ns-tabs .p-TabBar-tab.p-mod-current,
.ns-tabs .lm-TabBar-tab.lm-mod-current,
.ns-card .p-TabBar-tab.p-mod-current,
.ns-card .lm-TabBar-tab.lm-mod-current {
    background-color: var(--ns-bg) !important;
    color: var(--ns-accent) !important;
    border-bottom: 2px solid var(--ns-accent) !important;
}

.ns-tabs .p-TabBar-tab:hover,
.ns-tabs .lm-TabBar-tab:hover,
.ns-card .p-TabBar-tab:hover,
.ns-card .lm-TabBar-tab:hover {
    background-color: var(--ns-accent-soft) !important;
    color: var(--ns-accent) !important;
}

.ns-tabs .p-TabBar-tabLabel,
.ns-tabs .lm-TabBar-tabLabel,
.ns-tabs .p-TabBar-tabIcon,
.ns-tabs .lm-TabBar-tabIcon,
.ns-tabs .p-TabBar-tabCloseIcon,
.ns-tabs .lm-TabBar-tabCloseIcon,
.ns-card .p-TabBar-tabLabel,
.ns-card .lm-TabBar-tabLabel,
.ns-card .p-TabBar-tabIcon,
.ns-card .lm-TabBar-tabIcon,
.ns-card .p-TabBar-tabCloseIcon,
.ns-card .lm-TabBar-tabCloseIcon {
    color: inherit !important;
}

.ns-tabs .p-TabPanel-stackedPanel,
.ns-tabs .lm-TabPanel-stackedPanel,
.ns-tabs .p-StackedPanel,
.ns-tabs .lm-StackedPanel,
.ns-card .p-TabPanel-stackedPanel,
.ns-card .lm-TabPanel-stackedPanel,
.ns-card .p-StackedPanel,
.ns-card .lm-StackedPanel {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    padding: 0 !important;
    color-scheme: light !important;
}

.ns-tab-panel,
.widget-box.ns-tab-panel,
.widget-vbox.ns-tab-panel,
.widget-hbox.ns-tab-panel,
.ns-tab-panel.widget-box,
.ns-tab-panel.widget-vbox,
.ns-tab-panel.widget-hbox,
.ns-tab-panel.p-Widget,
.ns-tab-panel.lm-Widget {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    padding: 12px !important;
    color-scheme: light !important;
}

.ns-tab-panel .widget-box,
.ns-tab-panel .widget-vbox,
.ns-tab-panel .widget-hbox,
.ns-tab-panel .widget-gridbox,
.ns-tab-panel .widget-label,
.ns-tab-panel .widget-label-basic,
.ns-tab-panel .widget-html,
.ns-tab-panel .widget-html-content,
.ns-tab-panel .widget-output,
.ns-tab-panel .jupyter-widgets,
.ns-tab-panel .p-Widget,
.ns-tab-panel .lm-Widget {
    background-color: transparent !important;
    color: var(--ns-text) !important;
    color-scheme: light !important;
}

.ns-tab-row {
    background-color: transparent !important;
    color: var(--ns-text) !important;
    border-bottom: 1px solid var(--ns-border-soft);
    padding: 6px 0;
    color-scheme: light !important;
}

.ns-tab-row:last-child {
    border-bottom: none;
}

.ns-tabs select,
.ns-tabs input,
.ns-tabs textarea,
.ns-tabs button,
.ns-tab-panel select,
.ns-tab-panel input,
.ns-tab-panel textarea,
.ns-tab-panel button {
    color-scheme: light !important;
}

.ns-tabs select,
.ns-tabs input,
.ns-tabs textarea,
.ns-tab-panel select,
.ns-tab-panel input,
.ns-tab-panel textarea {
    background-color: var(--ns-bg) !important;
    color: var(--ns-text) !important;
    border-color: var(--ns-border) !important;
}

.ns-tabs select:disabled,
.ns-tabs input:disabled,
.ns-tabs textarea:disabled,
.ns-tab-panel select:disabled,
.ns-tab-panel input:disabled,
.ns-tab-panel textarea:disabled {
    background-color: var(--ns-bg-soft) !important;
    color: var(--ns-text-soft) !important;
    border-color: var(--ns-border-soft) !important;
    opacity: 1 !important;
}

/* ---------------------------------------------------------
   Progress / tqdm widget hard fix
   Fixes pale text such as:
   "Segmenting 2D slices (cellpose)"
--------------------------------------------------------- */

/* Output containers should not inherit Colab dark-mode color logic */
.jp-OutputArea,
.output_area,
.output_subarea,
.cell-output,
div.output,
div.output_wrapper,
div.output_area {
    color-scheme: light !important;
}

/* tqdm.notebook usually creates HBox + FloatProgress + HTML widgets */
.jp-OutputArea .jupyter-widgets,
.output_area .jupyter-widgets,
.output_subarea .jupyter-widgets,
.cell-output .jupyter-widgets,
.jupyter-widgets.widget-hbox,
.jupyter-widgets.widget-vbox,
.jupyter-widgets.widget-box,
.jupyter-widgets.widget-inline-hbox,
.widget-hbox,
.widget-vbox,
.widget-box,
.widget-inline-hbox {
    color-scheme: light !important;
    color: var(--ns-text) !important;
    opacity: 1 !important;
}

/* Strongly force all visible tqdm text */
.jp-OutputArea .jupyter-widgets .widget-label,
.jp-OutputArea .jupyter-widgets .widget-label-basic,
.jp-OutputArea .jupyter-widgets .widget-readout,
.jp-OutputArea .jupyter-widgets .widget-html,
.jp-OutputArea .jupyter-widgets .widget-html-content,
.output_area .jupyter-widgets .widget-label,
.output_area .jupyter-widgets .widget-label-basic,
.output_area .jupyter-widgets .widget-readout,
.output_area .jupyter-widgets .widget-html,
.output_area .jupyter-widgets .widget-html-content,
.output_subarea .jupyter-widgets .widget-label,
.output_subarea .jupyter-widgets .widget-label-basic,
.output_subarea .jupyter-widgets .widget-readout,
.output_subarea .jupyter-widgets .widget-html,
.output_subarea .jupyter-widgets .widget-html-content,
.cell-output .jupyter-widgets .widget-label,
.cell-output .jupyter-widgets .widget-label-basic,
.cell-output .jupyter-widgets .widget-readout,
.cell-output .jupyter-widgets .widget-html,
.cell-output .jupyter-widgets .widget-html-content,
.jupyter-widgets .widget-label,
.jupyter-widgets .widget-label-basic,
.jupyter-widgets .widget-readout,
.jupyter-widgets .widget-html,
.jupyter-widgets .widget-html-content,
.widget-hbox .widget-label,
.widget-hbox .widget-label-basic,
.widget-hbox .widget-readout,
.widget-hbox .widget-html,
.widget-hbox .widget-html-content,
.widget-inline-hbox .widget-label,
.widget-inline-hbox .widget-label-basic,
.widget-inline-hbox .widget-readout,
.widget-inline-hbox .widget-html,
.widget-inline-hbox .widget-html-content {
    color: var(--ns-text) !important;
    fill: var(--ns-text) !important;
    opacity: 1 !important;
    text-shadow: none !important;
}

/* tqdm timing/readout text can be slightly softer but must stay visible */
.jupyter-widgets .widget-readout,
.widget-hbox .widget-readout,
.widget-inline-hbox .widget-readout {
    color: var(--ns-text-muted) !important;
    opacity: 1 !important;
}

/* Some tqdm labels are on the progress widget itself */
.widget-progress,
.jupyter-widgets.widget-progress,
.widget-progress .widget-label,
.widget-progress .widget-label-basic,
.widget-progress .widget-readout {
    color: var(--ns-text) !important;
    opacity: 1 !important;
    color-scheme: light !important;
}

/* Prevent disabled/faded progress widgets from becoming unreadable */
.widget-progress:disabled,
.widget-progress[disabled],
.jupyter-widgets:disabled,
.jupyter-widgets[disabled],
.widget-hbox:disabled,
.widget-hbox[disabled] {
    opacity: 1 !important;
}

.widget-progress:disabled .widget-label,
.widget-progress:disabled .widget-label-basic,
.widget-progress:disabled .widget-readout,
.widget-progress[disabled] .widget-label,
.widget-progress[disabled] .widget-label-basic,
.widget-progress[disabled] .widget-readout {
    color: var(--ns-text) !important;
    opacity: 1 !important;
}

/* Progress bar track */
.widget-progress,
.widget-progress progress,
progress {
    background-color: var(--ns-bg-soft) !important;
    color-scheme: light !important;
}

/* Progress bar fill for browser progress element */
.widget-progress progress::-webkit-progress-value,
progress::-webkit-progress-value {
    background-color: var(--ns-accent) !important;
}

.widget-progress progress::-moz-progress-bar,
progress::-moz-progress-bar {
    background-color: var(--ns-accent) !important;
}

/* Progress bar fill for bootstrap-style bars */
.widget-progress .progress-bar,
.progress-bar {
    background-color: var(--ns-accent) !important;
}

/* ---------------------------------------------------------
   Animation
--------------------------------------------------------- */

@keyframes ns-spin {
    0% {
        transform: rotate(0deg);
    }
    100% {
        transform: rotate(360deg);
    }
}
</style>
"""

# ----------------------------------------
# Temporal functions for logging

import logging

class OutputWidgetHandler(logging.Handler):
    def __init__(self, output_widget):
        super().__init__()
        self.output_widget = output_widget

    def emit(self, record):
        msg = self.format(record)
        with self.output_widget:
            print(msg)

def configure_notebook_logging(output_widget):
    logger = logging.getLogger("nucleisky2d")
    logger.setLevel(logging.INFO)

    # Avoid duplicate handlers if the cell is run multiple times
    logger.handlers = [
        h for h in logger.handlers
        if not isinstance(h, OutputWidgetHandler)
    ]

    handler = OutputWidgetHandler(output_widget)
    handler.setLevel(logging.INFO)
    handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))

    logger.addHandler(handler)
    logger.propagate = False
    return logger


def _status_message(lines, title="Status", kind="info"):
    """Return a full-width styled status message."""
    if isinstance(lines, str):
        lines = [lines]

    safe_lines = "<br>".join(html.escape(str(x)) for x in lines)

    class_map = {
        "ok": "ns-message-ok",
        "error": "ns-message-error",
        "warn": "ns-message-warn",
        "info": "ns-message-warn",
    }

    cls = class_map.get(kind, "ns-message-warn")

    return f"""
    <div class="ns-message {cls}">
      <div class="ns-message-title">{html.escape(str(title))}</div>
      <div class="ns-message-body">{safe_lines}</div>
    </div>
    """


def _status_ok(lines, title="Ready"):
    return _status_message(lines, title=title, kind="ok")


def _status_err(lines, title="Action required"):
    return _status_message(lines, title=title, kind="error")


def _status_warn(lines, title="Please check"):
    return _status_message(lines, title=title, kind="warn")


def _status_info(lines, title="Working"):
    return f"""
    <div class="ns-status ns-status-running">
      <div class="ns-spinner"></div>
      <div>{html.escape(str(title))}: {html.escape(str(lines))}</div>
    </div>
    """

print("✅ Environment Ready")

# Step 1: Load your volumes and extract features
---

### **Select Volumes & Configuration**

In this step, you define the input data for the 3D matching process:

- **Reference Volume** (the “sky map”)
- **Moving Crop Volume** (the “telescope snapshot”) — either loaded from disk or synthesized as a random subvolume for testing.

**1. Source Volumes**

* `Full Path`: Path to your large reference Z-stack TIFF (or similar).
* `Load External Crop`: Use a real experimental crop volume you want to locate.
* `Generate Random Crop`: Synthesize a crop from the reference volume (useful for testing without a separate crop file).

**2. Random Generator Settings**
*(Only active if "Generate Random Crop" is selected)*

* `Output Size (Z,Y,X)`: Shape of the synthetic crop in voxels.
* `Scaling Range`: Simulates magnification differences (e.g., `0.7`–`1.3` allows the crop to be ~30% smaller or larger than the reference).
* `Fixed Seed`: Ensures the exact same random crop is generated every time.

**⚠️ The Golden Rule: Voxel Size (µm/px in Z,Y,X)**

NucleiSky3D matches in physical units (µm), so voxel sizes must be correct.

* The notebook will attempt to read voxel sizes from TIFF metadata.
* If metadata is missing or incorrect, manually enter voxel sizes for the full and crop volumes.

**Note on Dimension Order**

All NucleiSky3D APIs expect volumes in **ZYX** order.


In [ ]:
#@title Run to load image choosing GUI

# ----------------------------
# UI Styling
# ----------------------------
style_html = widgets.HTML(STYLE_CSS)

# ----------------------------
# Utilities
# ----------------------------

def _desc_width(*ws, w="90px"):
    for x in ws:
        try: x.style.description_width = w
        except: pass

def _mip_xy(vol_zyx: np.ndarray) -> np.ndarray: return np.max(np.asarray(vol_zyx), axis=0)
def _mip_xz(vol_zyx: np.ndarray) -> np.ndarray: return np.max(np.asarray(vol_zyx), axis=1)
def _mip_yz(vol_zyx: np.ndarray) -> np.ndarray: return np.max(np.asarray(vol_zyx), axis=2)



# ------------------------------------------------------------
# State & Globals
# ------------------------------------------------------------
state = dict(
    confirmed_full=False,
    confirmed_crop=False,
    last_full_path="",
    last_crop_path="",
    need_full=False,
    need_crop=False,
    full_header=None,
    crop_header=None,
    full_axes="",
    crop_axes="",
)

# Prefill ZYX
_lfz = float(globals().get("_NS3D_LAST_FULL_Z", 1.0))
_lfy = float(globals().get("_NS3D_LAST_FULL_Y", 1.0))
_lfx = float(globals().get("_NS3D_LAST_FULL_X", 1.0))
_lcz = float(globals().get("_NS3D_LAST_CROP_Z", 1.0))
_lcy = float(globals().get("_NS3D_LAST_CROP_Y", 1.0))
_lcx = float(globals().get("_NS3D_LAST_CROP_X", 1.0))

# ----------------------------
# WIDGETS CONSTRUCTION
# ----------------------------
title_html = widgets.HTML(
    "<div class='ns-page-title'>Choose 3D Volumes</div>"
    "<div class='ns-page-subtitle'>Select your reference 3D volume and define the crop region.</div>"
)

# --- Card 1: Main Inputs ---
full_path = widgets.Text(
    value='', description='Full Path',
    placeholder='/path/to/large_volume.tif', layout=widgets.Layout(width='98%')
)
full_channel_axis = widgets.Dropdown(
    options=[("Select axis", None)], value=None, description="Full ch axis",
    layout=widgets.Layout(width='220px')
)
full_channel_index = widgets.Dropdown(
    options=[("Select index", None)], value=None, description="Full ch idx",
    layout=widgets.Layout(width='220px')
)
full_channel_box = widgets.HBox([full_channel_axis, full_channel_index], layout=widgets.Layout(display='none', gap='10px'))

mode = widgets.ToggleButtons(
    options=[("Load External Crop", "external"), ("Generate Random Crop", "random")],
    value="external", layout=widgets.Layout(width='auto'), style={'button_width':'180px'}
)
crop_path = widgets.Text(value='', description='Crop Path', placeholder='/path/to/crop_volume.tif', layout=widgets.Layout(width='98%'))
crop_channel_axis = widgets.Dropdown(
    options=[("Select axis", None)], value=None, description="Crop ch axis",
    layout=widgets.Layout(width='220px')
)
crop_channel_index = widgets.Dropdown(
    options=[("Select index", None)], value=None, description="Crop ch idx",
    layout=widgets.Layout(width='220px')
)
crop_channel_box = widgets.HBox([crop_channel_axis, crop_channel_index], layout=widgets.Layout(display='none', gap='10px'))

# Optional output folder. If left empty, a timestamped folder is created automatically.
output_dir = widgets.Text(
    value='',
    description='Folder',
    placeholder='/path/to/output_folder',
    layout=widgets.Layout(width='98%')
)

input_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Source Volumes</div>"),
    full_path,
    full_channel_box,
    widgets.HTML("<div style='height:8px'></div>"),
    widgets.HTML("<div class='ns-label'>Crop Strategy</div>"),
    mode,
    widgets.HTML("<div style='height:8px'></div>"),
    crop_path,
    crop_channel_box
])
input_card.add_class("ns-card")

# --- Card 2: Output Folder ---
output_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Output Folder</div>"),
    widgets.HTML(
        "<div class='ns-desc'>Choose where results should be saved, "
        "or leave it empty to create a default folder automatically.</div>"
    ),
    output_dir,
])
output_card.add_class("ns-card")

# --- Card 3: Random Settings ---
patch_z = widgets.IntText(value=64, description='Z (vox)', layout=widgets.Layout(width='98%'))
patch_y = widgets.IntText(value=128, description='Y (vox)', layout=widgets.Layout(width='98%'))
patch_x = widgets.IntText(value=128, description='X (vox)', layout=widgets.Layout(width='98%'))

scale_min = widgets.FloatText(value=1.0, description='Scale Min', layout=widgets.Layout(width='98%'))
scale_max = widgets.FloatText(value=1.0, description='Scale Max', layout=widgets.Layout(width='98%'))

use_seed = widgets.Checkbox(value=False, description='Fixed Seed', indent=False)
rng_seed = widgets.IntText(value=42, description='Value', layout=widgets.Layout(width='120px'), disabled=True)

rand_grid = widgets.VBox([
    widgets.HTML("<div class='ns-header'>3. Random Generator Settings</div>"),
    widgets.HTML("<div class='ns-label'>Output Size (Voxels)</div>"),
    widgets.HBox([patch_z, patch_y, patch_x], layout=widgets.Layout(gap="10px")),
    widgets.HTML("<div class='ns-label'>Scaling Range</div>"),
    widgets.HBox([scale_min, scale_max], layout=widgets.Layout(gap="10px")),
    widgets.HTML("<div style='height:12px'></div>"),
    widgets.HBox([use_seed, rng_seed], layout=widgets.Layout(gap="10px")),
    widgets.HTML("<div style='font-size:12px; color:#64748B;'>Note: 3D random crops are automatically generated with a random SO(3) rotation.</div>"),
])
rand_card = widgets.VBox([rand_grid])
rand_card.add_class("ns-card")

# --- Card 3: Manual Metadata Warning ---
manual_msg = widgets.HTML("")
vox_full_z = widgets.FloatText(value=_lfz, description='Full Z (µm)', layout=widgets.Layout(width='180px'))
vox_full_y = widgets.FloatText(value=_lfy, description='Full Y (µm)', layout=widgets.Layout(width='180px'))
vox_full_x = widgets.FloatText(value=_lfx, description='Full X (µm)', layout=widgets.Layout(width='180px'))

vox_crop_z = widgets.FloatText(value=_lcz, description='Crop Z (µm)', layout=widgets.Layout(width='180px'))
vox_crop_y = widgets.FloatText(value=_lcy, description='Crop Y (µm)', layout=widgets.Layout(width='180px'))
vox_crop_x = widgets.FloatText(value=_lcx, description='Crop X (µm)', layout=widgets.Layout(width='180px'))

box_full = widgets.HBox([vox_full_z, vox_full_y, vox_full_x], layout=widgets.Layout(width="100%", margin="5px 0"))
box_crop = widgets.HBox([vox_crop_z, vox_crop_y, vox_crop_x], layout=widgets.Layout(width="100%", margin="5px 0"))

manual_card = widgets.VBox([
    widgets.HTML("<div class='ns-header' style='color:#991B1B; border-color:#FECACA;'>Missing Metadata</div>"),
    manual_msg,
    box_full,
    box_crop
])
manual_card.add_class("ns-card")
manual_card.add_class("ns-card-warn")
manual_card.layout.display = "none"

# Outputs
status_html = widgets.HTML("", layout=widgets.Layout(width="100%"))
run_button = widgets.Button(description='Load & Process', button_style='primary', icon='play', layout=widgets.Layout(width='100%', height='40px'))
run_button.add_class('ns-primary-button')
fig_out = widgets.Output()

_desc_width(full_path, crop_path, output_dir, full_channel_axis, full_channel_index, w="100px")
_desc_width(patch_z, patch_y, patch_x, scale_min, scale_max, w="80px")

# ----------------------------
# Logic & Sync
# ----------------------------
def _shape_ndim(header):
    shape = tuple(header.get("shape") or ())
    return shape, len(shape)

def _configure_channel_controls(header, axis_widget, index_widget, box_widget):
    shape, ndim = _shape_ndim(header)
    if ndim == 4:
        axis_opts = [(f"axis {ax} (size={shape[ax]})", ax) for ax in range(4)]
        axis_widget.options = [("Select axis", None)] + axis_opts
        if axis_widget.value not in [v for _, v in axis_widget.options]:
            axis_widget.value = None
        box_widget.layout.display = "flex"
    else:
        axis_widget.options = [("N/A for 3D", None)]
        axis_widget.value = None
        index_widget.options = [("N/A for 3D", None)]
        index_widget.value = None
        box_widget.layout.display = "none"

def _refresh_channel_index(axis_widget, index_widget, header):
    shape, ndim = _shape_ndim(header)
    if ndim != 4 or axis_widget.value is None:
        index_widget.options = [("Select index", None)]
        index_widget.value = None
        return
    axis = int(axis_widget.value)
    axis_size = int(shape[axis])
    opts = [(str(i), i) for i in range(axis_size)]
    index_widget.options = [("Select index", None)] + opts
    if index_widget.value not in [v for _, v in index_widget.options]:
        index_widget.value = None

def _inspect_and_configure(path_str, label, *, axis_widget, index_widget, box_widget):
    header = inspect_volume_header(path_str)
    shape, ndim = _shape_ndim(header)
    if ndim not in (3, 4):
        raise ValueError(f"{label} volume must be 3D or 4D, got shape {shape}.")
    _configure_channel_controls(header, axis_widget, index_widget, box_widget)
    _refresh_channel_index(axis_widget, index_widget, header)
    return header

def _resolve_channel_selection(header, label, *, axis_widget, index_widget):
    shape, ndim = _shape_ndim(header)
    if ndim == 3:
        return None, 0
    if axis_widget.value is None or index_widget.value is None:
        raise ValueError(f"{label} volume is 4D; select channel axis and channel index.")
    axis = int(axis_widget.value)
    index = int(index_widget.value)
    axis_size = int(shape[axis])
    if index < 0 or index >= axis_size:
        raise ValueError(f"{label} channel index {index} is out of bounds for axis {axis} (size={axis_size}).")
    return axis, index

def _sync_ui():
    use_rand = (mode.value == "random")
    crop_path.layout.display = "none" if use_rand else "flex"
    crop_channel_box.layout.display = "none" if use_rand else crop_channel_box.layout.display
    rand_card.layout.display = "flex" if use_rand else "none"
    rng_seed.disabled = not use_seed.value

    show_manual = state["need_full"] or state["need_crop"]
    manual_card.layout.display = "flex" if show_manual else "none"
    box_full.layout.display = "flex" if state["need_full"] else "none"
    box_crop.layout.display = "flex" if state["need_crop"] else "none"

def _on_paths_changed(_=None):
    fp = full_path.value.strip()
    cp = crop_path.value.strip()

    if fp != state["last_full_path"]:
        state["confirmed_full"] = False
        state["last_full_path"] = fp
        state["full_header"] = None
    if cp != state["last_crop_path"]:
        state["confirmed_crop"] = False
        state["last_crop_path"] = cp
        state["crop_header"] = None

    state["need_full"] = False
    state["need_crop"] = False
    manual_msg.value = ""

    if fp and Path(fp).exists():
        state["full_header"] = _inspect_and_configure(
            fp, "Full", axis_widget=full_channel_axis, index_widget=full_channel_index, box_widget=full_channel_box
        )
    else:
        full_channel_box.layout.display = "none"

    use_rand = (mode.value == "random")
    if (not use_rand) and cp and Path(cp).exists():
        state["crop_header"] = _inspect_and_configure(
            cp, "Crop", axis_widget=crop_channel_axis, index_widget=crop_channel_index, box_widget=crop_channel_box
        )
    else:
        crop_channel_box.layout.display = "none"

    _sync_ui()

full_path.observe(_on_paths_changed, names="value")
crop_path.observe(_on_paths_changed, names="value")
mode.observe(lambda _: (_on_paths_changed(), _sync_ui()), names="value")
use_seed.observe(lambda _: _sync_ui(), names="value")
full_channel_axis.observe(lambda _: _refresh_channel_index(full_channel_axis, full_channel_index, state.get("full_header") or {}), names="value")
crop_channel_axis.observe(lambda _: _refresh_channel_index(crop_channel_axis, crop_channel_index, state.get("crop_header") or {}), names="value")

# ----------------------------
# Main Handler
# ----------------------------
def on_run_clicked(_):
    global img_full, img_crop, voxel_size_full_um, voxel_size_crop_um, RESULT_DIR, GROUND_TRUTH
    global FULL_PATH_USED, CROP_PATH_USED

    status_html.value = _status_info("Loading and preparing 3D volumes.", title="Processing")
    with fig_out: clear_output(wait=True)

    try:
        _on_paths_changed()
        full_path_str = full_path.value.strip()
        use_rand = (mode.value == "random")

        if not full_path_str:
            raise FileNotFoundError("Please provide a Full Path.")
        if not use_rand:
            crop_path_str = crop_path.value.strip()
            if not crop_path_str or not Path(crop_path_str).exists():
                raise FileNotFoundError("Crop image not found.")

        # --- Load Full ---
        missing = []

        if Path(full_path_str).exists():
            full_header = state.get("full_header") or _inspect_and_configure(
                full_path_str, "Full", axis_widget=full_channel_axis, index_widget=full_channel_index, box_widget=full_channel_box
            )
            full_axis, full_index = _resolve_channel_selection(
                full_header, "Full", axis_widget=full_channel_axis, index_widget=full_channel_index
            )
            img_full_load = load_volume(full_path_str, channel_axis=full_axis, channel_index=full_index)
            if np.asarray(img_full_load).ndim != 3:
                raise ValueError(f"Full volume did not resolve to 3D ZYX. Got shape {np.asarray(img_full_load).shape}.")
            state["full_axes"] = "ZYX"
            try:
                vox_meta = require_voxel_size_um_zyx(full_path_str)
            except ValueError:
                if state["confirmed_full"]:
                    manual_full = (float(vox_full_z.value), float(vox_full_y.value), float(vox_full_x.value))
                    vox_meta = require_voxel_size_um_zyx(full_path_str, fallback=manual_full)
                else:
                    missing.append("Full Volume")
                    state["need_full"] = True
                    vox_meta = None
        else:
            raise FileNotFoundError("Full volume not found.")

        # Voxel logic for Full
        if vox_meta is not None:
            vox_full = tuple(float(v) for v in vox_meta)
            pix_full_src = "metadata"
            state["need_full"] = False
            state["confirmed_full"] = False

        # --- Load/Gen Crop ---
        GROUND_TRUTH_TEMP = None
        if not use_rand:
            crop_header = state.get("crop_header") or _inspect_and_configure(
                crop_path_str, "Crop", axis_widget=crop_channel_axis, index_widget=crop_channel_index, box_widget=crop_channel_box
            )
            crop_axis, crop_index = _resolve_channel_selection(
                crop_header, "Crop", axis_widget=crop_channel_axis, index_widget=crop_channel_index
            )
            img_crop_load = load_volume(crop_path_str, channel_axis=crop_axis, channel_index=crop_index)
            if np.asarray(img_crop_load).ndim != 3:
                raise ValueError(f"Crop volume did not resolve to 3D ZYX. Got shape {np.asarray(img_crop_load).shape}.")
            state["crop_axes"] = "ZYX"
            try:
                vox_c_meta = require_voxel_size_um_zyx(crop_path_str)
            except ValueError:
                if state["confirmed_crop"]:
                    manual_crop = (float(vox_crop_z.value), float(vox_crop_y.value), float(vox_crop_x.value))
                    vox_c_meta = require_voxel_size_um_zyx(crop_path_str, fallback=manual_crop)
                else:
                    missing.append("Crop Volume")
                    state["need_crop"] = True
                    vox_c_meta = None

            if vox_c_meta is not None:
                vox_crop = tuple(float(v) for v in vox_c_meta)
                pix_crop_src = "metadata"
                state["need_crop"] = False
                state["confirmed_crop"] = False
        else:
            # Random bypasses metadata missing check because we derive it mathematically
            vox_crop = None
            state["crop_axes"] = "ZYX"

        # --- Evaluate Missing States ---
        if missing:
            if state["need_full"]: state["confirmed_full"] = True
            if state["need_crop"]: state["confirmed_crop"] = True
            manual_msg.value = f"<b>Voxel size not found (ZYX needed):</b> {', '.join(missing)}."
            status_html.value = ""
            _sync_ui()
            return

        # Output Prep
        out_dir = output_dir.value.strip()

        if out_dir:
            RESULT_DIR = str(Path(out_dir))
            Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)
            output_dir_msg = f"Using specified output directory: {RESULT_DIR}"
        else:
            RESULT_DIR = str(make_result_dir(big_image_path=full_path_str, tag="NucleiSky3D"))
            output_dir_msg = f"No output directory specified. Using auto-generated directory: {RESULT_DIR}"

        # Proceed
        if use_rand:
            crop_shape_zyx = (int(patch_z.value), int(patch_y.value), int(patch_x.value))

            # --- SCALE RANGE FIX ---
            s_min = float(scale_min.value)
            s_max = float(scale_max.value)

            if s_min <= 0:
                raise ValueError("Scale Min must be positive (> 0).")
            if s_max < s_min:
                raise ValueError("Scale Max cannot be smaller than Scale Min.")

            # If the user selects e.g. 1.0 to 1.0 (no scale variance), we add a tiny epsilon
            # to satisfy the strict (min < max) requirement in the generation utility.
            if s_min == s_max:
                s_max += 1e-5

            s_range = (s_min, s_max)

            rng = np.random.default_rng(int(rng_seed.value)) if use_seed.value else None

            img_crop_load, crop_vox_derived, GROUND_TRUTH_TEMP = generate_random_subvolume_3d(
                img_full_load, crop_shape_zyx=crop_shape_zyx, scale_range=s_range,
                voxel_size_um=vox_full, rng=rng
            )
            vox_crop = tuple(float(v) for v in np.asarray(crop_vox_derived).reshape(3,))
            pix_crop_src = "derived"

            # Save the randomly generated crop strictly to disk for downstream use
            CROP_PATH_USED = str(Path(RESULT_DIR) / "random_crop.tif")
            tifffile.imwrite(CROP_PATH_USED, np.asarray(img_crop_load))
        else:
            CROP_PATH_USED = crop_path_str

        # Apply globals
        img_full = img_full_load
        img_crop = img_crop_load
        voxel_size_full_um = vox_full
        voxel_size_crop_um = vox_crop
        GROUND_TRUTH = GROUND_TRUTH_TEMP
        FULL_PATH_USED = full_path_str

        # Save Manual States
        globals()["_NS3D_LAST_FULL_Z"], globals()["_NS3D_LAST_FULL_Y"], globals()["_NS3D_LAST_FULL_X"] = vox_full
        globals()["_NS3D_LAST_CROP_Z"], globals()["_NS3D_LAST_CROP_Y"], globals()["_NS3D_LAST_CROP_X"] = vox_crop

        state["need_full"] = False; state["need_crop"] = False
        manual_msg.value = ""
        _sync_ui()

        # Save Metadata Config
        with open(Path(RESULT_DIR) / "inputs_metadata.json", "w") as f:
            json.dump({
                "full_image_path": FULL_PATH_USED,
                "crop_image_path": CROP_PATH_USED,
                "voxel_size_full_um_zyx": voxel_size_full_um,
                "voxel_size_crop_um_zyx": voxel_size_crop_um,
                "mode": mode.value,
                "full_axes": state["full_axes"],
                "crop_axes": state["crop_axes"],
                "full_channel_axis": full_channel_axis.value,
                "full_channel_index": full_channel_index.value,
                "crop_channel_axis": crop_channel_axis.value,
                "crop_channel_index": crop_channel_index.value
            }, f, indent=2)

        s_lines = [
            "Success!",
            output_dir_msg,
            f"Working directory: {Path(RESULT_DIR).name}",
            f"Full shape ({state['full_axes']}): {tuple(img_full.shape)} | Voxel: {voxel_size_full_um} ({pix_full_src})",
            f"Crop shape ({state['crop_axes']}): {tuple(img_crop.shape)} | Voxel: {voxel_size_crop_um} ({pix_crop_src})"
        ]
        if GROUND_TRUTH is not None: s_lines.append("ℹ️ Random crop ground-truth available in `GROUND_TRUTH`.")
        status_html.value = _status_ok(s_lines)

        # Plot MIPs using safe rendering
        with fig_out:
            fig, axes = plt.subplots(2, 3, figsize=(10, 6), dpi=110)

            imshow_safe3d(axes[0,0], _mip_xy(img_full), title="Full MIP XY")
            imshow_safe3d(axes[0,1], _mip_xz(img_full), title="Full MIP XZ")
            imshow_safe3d(axes[0,2], _mip_yz(img_full), title="Full MIP YZ")

            imshow_safe3d(axes[1,0], _mip_xy(img_crop), title="Crop MIP XY")
            imshow_safe3d(axes[1,1], _mip_xz(img_crop), title="Crop MIP XZ")
            imshow_safe3d(axes[1,2], _mip_yz(img_crop), title="Crop MIP YZ")

            plt.tight_layout()
            plt.show()

    except Exception as e:
        status_html.value = _status_err([str(e)], title="Error during processing")
        with fig_out: traceback.print_exc()

run_button.on_click(on_run_clicked)

# ----------------------------
# Final Display
# ----------------------------
inner1 = widgets.VBox([
    title_html,
    input_card,
    output_card,
    rand_card,
    manual_card,
    run_button,
    widgets.Box([status_html], layout=widgets.Layout(width="100%", margin="10px 0")),
    fig_out
])
inner1.add_class("ns-root")

ui = widgets.VBox([
    style_html,
    inner1
], layout=widgets.Layout(width="90%", margin="0 auto"))

_sync_ui()
display(ui)

# Segmentation & Feature Extraction (3D)
---

In [ ]:
#@title Run to apply or load the 2.5D segmentation and feature extraction GUI

# ----------------------------
# UI Styling
# ----------------------------
style_html = widgets.HTML(STYLE_CSS)


def _status_message(lines, title="Status", kind="info"):
    """Return a full-width styled status message."""
    if isinstance(lines, str):
        lines = [lines]

    safe_lines = "<br>".join(html.escape(str(x)) for x in lines)

    class_map = {
        "ok": "ns-message-ok",
        "error": "ns-message-error",
        "warn": "ns-message-warn",
        "info": "ns-message-warn",
    }

    cls = class_map.get(kind, "ns-message-warn")

    return f"""
    <div class="ns-message {cls}">
      <div class="ns-message-title">{html.escape(str(title))}</div>
      <div class="ns-message-body">{safe_lines}</div>
    </div>
    """


def _status_ok(lines, title="Ready"):
    return _status_message(lines, title=title, kind="ok")


def _status_err(lines, title="Action required"):
    return _status_message(lines, title=title, kind="error")


def _status_warn(lines, title="Please check"):
    return _status_message(lines, title=title, kind="warn")


def _status_info(lines, title="Working"):
    return f"""
    <div class="ns-status ns-status-running">
      <div class="ns-spinner"></div>
      <div>{html.escape(str(title))}: {html.escape(str(lines))}</div>
    </div>
    """

# ----------------------------
# WIDGETS
# ----------------------------

# --- Top Level: Source Selection ---
source_mode = widgets.ToggleButtons(
    options=[('Run 2.5D Segmentation', 'new'), ('Load Existing Masks', 'existing')],
    value='new',
    style={'button_width': '220px'}
)

source_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Input Source</div>"),
    widgets.HTML("<div class='ns-desc'>Run a new 2.5D segmentation workflow or load existing label masks.</div>"),
    source_mode,
])
source_card.add_class("ns-card")

mask_path_full = widgets.Text(description="Labels Full", placeholder="/path/to/mask_full.tif", layout=widgets.Layout(width="98%"))
mask_path_crop = widgets.Text(description="Labels Crop", placeholder="/path/to/mask_crop.tif", layout=widgets.Layout(width="98%"))

mask_input_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Existing Mask Paths</div>"),
    mask_path_full, mask_path_crop
])
mask_input_card.add_class("ns-card")

# --- Step 1: Scale Matching ---
scale_strategy = widgets.Dropdown(options=["coarsest", "finest", "none"], value="coarsest", description="Scale Match:", layout=widgets.Layout(width="250px"), tooltip="Force volumes to same physical resolution before segmenting")
normalize_before = widgets.Checkbox(value=False, description="Percentile normalize before segmentation")

step1_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'><span class='ns-step'>Step 1.</span> 3D Scale Matching</div>"),
    widgets.HTML("<div style='font-size:12px; color:#64748B; margin-bottom:8px;'>Ensure the Full and Crop volumes are rescaled to the same physical µm/voxel resolution before running AI models.</div>"),
    widgets.HBox([scale_strategy]),
    normalize_before,
])
step1_card.add_class("ns-card")

# --- Step 2: 2D Segmentation Method ---
seg_method = widgets.Dropdown(
    options=[
        ("InstanSeg (Fast Deep Learning)", "instanseg"),
        ("Cellpose (Deep Learning)", "cellpose"),
        ("Auto-threshold (Classic CV)", "threshold"),
    ],
    value="instanseg",
    layout=widgets.Layout(width="98%")
)
method_desc = widgets.HTML("")

# InstanSeg Specifics
inst_model = widgets.Dropdown(options=["brightfield_nuclei", "fluorescence_nuclei_and_cells"], value="brightfield_nuclei", description="Model")
inst_target = widgets.Dropdown(options=["nuclei", "cells"], value="nuclei", description="Target")
inst_mode = widgets.Dropdown(options=[("Auto (small/medium)", "auto"), ("Small image", "small"), ("Medium (tiled)", "medium")], value="auto", description="Mode")
inst_clean = widgets.Checkbox(value=True, description="Cleanup Fragments")
inst_verb = widgets.IntSlider(value=0, min=0, max=2, description="Verbosity")

inst_adv_ui = widgets.VBox([inst_mode, inst_clean, inst_verb])
inst_adv_ui.add_class('ns-accordion-body')
inst_accord = widgets.Accordion(children=[inst_adv_ui], titles=('Advanced InstanSeg Settings',))
inst_accord.add_class('ns-accordion')

inst_panel = widgets.VBox([
    widgets.HTML("<div class='ns-label'>Model Configuration</div>"),
    widgets.HBox([inst_model, inst_target], layout=widgets.Layout(gap="20px")),
    widgets.HTML("<div style='height:8px'></div>"),
    inst_accord
])

# Threshold Specifics
thr_method = widgets.Dropdown(options=["otsu", "li", "yen", "triangle", "isodata"], value="otsu", description="Algorithm")
thr_sigma = widgets.FloatSlider(value=1.0, min=0, max=5, step=0.25, description="Blur Sigma")
thr_min_obj = widgets.IntText(value=80, description="Min Area (px)")
thr_watershed = widgets.Checkbox(value=True, description="Watershed Split")

thr_adv_ui = widgets.VBox([thr_sigma, widgets.HBox([thr_min_obj, thr_watershed])])
thr_adv_ui.add_class('ns-accordion-body')
thr_accord = widgets.Accordion(children=[thr_adv_ui], titles=('Advanced CV Parameters',))
thr_accord.add_class('ns-accordion')

thr_panel = widgets.VBox([
    widgets.HTML("<div class='ns-label'>Thresholding Configuration</div>"),
    thr_method,
    widgets.HTML("<div style='height:8px'></div>"),
    thr_accord
])

step2_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'><span class='ns-step'>Step 2.</span> 2D Segmentation Method</div>"),
    widgets.HTML("<div style='font-size:12px; color:#64748B; margin-bottom:8px;'>Choose the algorithm used to segment each individual 2D Z-slice.</div>"),
    seg_method,
    method_desc,
    inst_panel,
    thr_panel
])
step2_card.add_class("ns-card")

# --- Step 3: 3D Mask Linking ---
stitch_iou = widgets.FloatSlider(value=0.3, min=0.01, max=0.99, step=0.05, description="Z-Stitch IoU:", layout=widgets.Layout(width="350px"))

step3_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'><span class='ns-step'>Step 3.</span> 3D Mask Linking</div>"),
    widgets.HTML("<div style='font-size:12px; color:#64748B; margin-bottom:8px;'>Set the Intersection-over-Union (IoU) threshold required to link a 2D nucleus in slice Z to the same nucleus in slice Z+1.</div>"),
    stitch_iou
])
step3_card.add_class("ns-card")

# Group the 3 steps
segmentation_workflow = widgets.VBox([step1_card, step2_card, step3_card])

# --- Outputs (Stacked vertically to prevent overlapping) ---
run_btn = widgets.Button(description="Run 2.5D Segment + Extract", button_style="success", icon="cogs", layout=widgets.Layout(width="320px", height="40px"))
run_btn.add_class("ns-primary-button")
status_html = widgets.HTML("", layout=widgets.Layout(width="100%"))
out_log = widgets.Output(layout=widgets.Layout(max_height="200px", overflow="auto", border="1px solid #E2E8F0", padding="8px", background_color="#F8FAFC"))
out_fig = widgets.Output()
out_tbl = widgets.Output(layout=widgets.Layout(max_height="250px", overflow="auto"))

# ----------------------------
# LOGIC & BINDINGS
# ----------------------------
def _update_ui_state(_=None):
    is_existing = (source_mode.value == 'existing')

    # Toggles
    mask_input_card.layout.display = "flex" if is_existing else "none"
    segmentation_workflow.layout.display = "none" if is_existing else "flex"
    run_btn.description = "Load Masks & Extract" if is_existing else "Run 2.5D Segment + Extract"

    # Methods
    m = seg_method.value
    inst_panel.layout.display = "flex" if m == "instanseg" else "none"
    thr_panel.layout.display = "flex" if m == "threshold" else "none"

    if m == "cellpose": method_desc.value = "<i>Standard deep learning model for cellular segmentation.</i>"
    elif m == "instanseg": method_desc.value = "<i>High-speed pytorch implementation. Best for large 3D datasets.</i>"
    elif m == "threshold": method_desc.value = "<i>Classical computer vision. Sliced-based thresholding.</i>"

source_mode.observe(_update_ui_state, names="value")
seg_method.observe(_update_ui_state, names="value")
_update_ui_state()

# ----------------------------
# EXECUTION HANDLER
# ----------------------------
def on_run(_):
    # Map to globals required by Step 3
    global img_full_seg, img_crop_seg, labels_full_seg, labels_crop_seg
    global voxel_full_seg, voxel_crop_seg
    global df_full, df_crop

    # 1. Reset Displays
    status_html.value = _status_info("Running segmentation and feature extraction.", title="Processing")
    out_log.clear_output(wait=True)
    out_fig.clear_output(wait=True)
    out_tbl.clear_output(wait=True)

    # 2. Run Main Processing inside out_log to catch all prints/progress bars cleanly
    with out_log:
        try:
            if "img_full" not in globals() or "img_crop" not in globals():
                raise RuntimeError("Missing raw image variables. Please run Step 1 first.")

            PIXEL_SIZE_FULL_UM_ZYX = tuple(float(v) for v in voxel_size_full_um)
            PIXEL_SIZE_CROP_UM_ZYX = tuple(float(v) for v in voxel_size_crop_um)

            RESULT_DIR_LOCAL = Path(globals().get("RESULT_DIR", "./NucleiSky3D_Results"))
            SEG_DIR = RESULT_DIR_LOCAL / "segmentation"
            SEG_DIR.mkdir(parents=True, exist_ok=True)

            if source_mode.value == "existing":
                print("📂 Loading existing masks...")
                labels_full_local = load_volume(mask_path_full.value.strip())
                labels_crop_local = load_volume(mask_path_crop.value.strip())

                img_full_seg, img_crop_seg = img_full, img_crop
                labels_full_seg, labels_crop_seg = labels_full_local, labels_crop_local
                voxel_full_seg, voxel_crop_seg = PIXEL_SIZE_FULL_UM_ZYX, PIXEL_SIZE_CROP_UM_ZYX
                seg_src = "Loaded existing labels (bypassing scaling)"

            else:
                vol_full = ij_percentile_normalize(img_full) if normalize_before.value else img_full
                vol_crop = ij_percentile_normalize(img_crop) if normalize_before.value else img_crop
                chosen_method = seg_method.value

                # --- SCALE NORMALIZATION ---
                if scale_strategy.value != "none":
                    print(f"⚖️ Normalizing voxel sizes (strategy: {scale_strategy.value})...")
                    (
                        vol_full_rescaled, vol_crop_rescaled,
                        vox_full_rescaled, vox_crop_rescaled,
                        _, _, target_req
                    ) = scale_normalize_pair_for_segmentation(
                        vol_full, vol_crop,
                        PIXEL_SIZE_FULL_UM_ZYX, PIXEL_SIZE_CROP_UM_ZYX,
                        strategy=scale_strategy.value
                    )
                    print(f"   Target common resolution: {tuple(np.round(target_req, 3))} µm/voxel")
                else:
                    vol_full_rescaled, vol_crop_rescaled = vol_full, vol_crop
                    vox_full_rescaled, vox_crop_rescaled = PIXEL_SIZE_FULL_UM_ZYX, PIXEL_SIZE_CROP_UM_ZYX

                # --- BUILD SETTINGS ---
                settings = {"method": chosen_method}
                if chosen_method == "instanseg":
                    settings["instanseg"] = {
                        "model_name": inst_model.value, "target": inst_target.value,
                        "mode": inst_mode.value, "verbosity": inst_verb.value,
                        "cleanup_fragments": inst_clean.value
                    }
                elif chosen_method == "threshold":
                    settings["threshold"] = {
                        "method": thr_method.value,
                        "gaussian_sigma_zyx": (0.0, thr_sigma.value, thr_sigma.value),
                        "min_object_size_vox": thr_min_obj.value,
                        "do_watershed": thr_watershed.value
                    }
                elif chosen_method == "cellpose":
                    settings["cellpose"] = {} # Defaults

                # --- SEGMENT ---
                print(f"\n🧠 Segmenting FULL volume ({chosen_method})...")
                lbl_full_rescaled = segment_nuclei_2p5d(
                    vol_full_rescaled, method=chosen_method,
                    pixel_size_um_zyx=vox_full_rescaled, settings=settings, min_iou=stitch_iou.value
                )

                print("\n🧠 Segmenting CROP volume...")
                lbl_crop_rescaled = segment_nuclei_2p5d(
                    vol_crop_rescaled, method=chosen_method,
                    pixel_size_um_zyx=vox_crop_rescaled, settings=settings, min_iou=stitch_iou.value
                )

                img_full_seg, img_crop_seg = vol_full_rescaled, vol_crop_rescaled
                labels_full_seg, labels_crop_seg = lbl_full_rescaled, lbl_crop_rescaled
                voxel_full_seg, voxel_crop_seg = vox_full_rescaled, vox_crop_rescaled

                seg_src = f"2.5D Seg (Method: {chosen_method}, Linker IoU: {stitch_iou.value})"

                # --- SAVE MASKS ---
                print("\n💾 Saving segmentation masks to disk...")
                save_tiff_zyx(SEG_DIR / "labels_full.tif", labels_full_seg.astype(np.int32), voxel_size_um_zyx=voxel_full_seg, compress="zlib")
                save_tiff_zyx(SEG_DIR / "labels_crop.tif", labels_crop_seg.astype(np.int32), voxel_size_um_zyx=voxel_crop_seg, compress="zlib")

            # --- FEATURE EXTRACTION ---
            print("\n📊 Extracting 3D morphological features...")
            df_full = extract_nuclear_features_3d(labels_full_seg, pixel_size_um=voxel_full_seg)
            df_crop = extract_nuclear_features_3d(labels_crop_seg, pixel_size_um=voxel_crop_seg)

            # Save CSVs
            print("💾 Saving feature tables...")
            df_full.to_csv(SEG_DIR / "features_full.csv", index=False)
            df_crop.to_csv(SEG_DIR / "features_crop.csv", index=False)
            print("✅ Pipeline execution successful.")

            # Update success HTML
            status_html.value = _status_ok([
                f"✅ {seg_src}",
                f"Saved to: {SEG_DIR.name}/",
                f"Full volume nuclei: {len(df_full)} | Crop volume nuclei: {len(df_crop)}"
            ])

        except Exception as e:
            status_html.value = _status_err([str(e)], title="Segmentation Failed")
            traceback.print_exc()
            return # Exit function on error to prevent plotting failures

    # 3. Plotting & Tables (Only runs if processing succeeds)
    with out_fig:
        fig, axes = plt.subplots(2, 2, figsize=(12, 10), dpi=110)

        def plot_slice_fast(ax_raw, ax_overlay, img_vol, df, title_prefix):
            # Auto-select the Z-slice that contains the most nuclei
            if df is not None and not df.empty and "centroid_z_px" in df.columns:
                best_z = int(df["centroid_z_px"].round().value_counts().idxmax())
            else:
                best_z = img_vol.shape[0] // 2

            best_z = np.clip(best_z, 0, img_vol.shape[0] - 1)
            raw_slice = img_vol[best_z]

            # Render raw base image
            disp_raw = imshow_safe3d(ax_raw, raw_slice, title=f"{title_prefix} Raw (Z={best_z})", max_dim=1500)
            imshow_safe3d(ax_overlay, raw_slice, title=f"{title_prefix} Nuclei Identified (Z={best_z})", max_dim=1500)

            # Overlay ultra-fast point scatter over the downsampled image
            if df is not None and not df.empty and "centroid_z_px" in df.columns:
                # Get points within +/- 1.5 pixels of the target Z slice
                z_mask = np.abs(df["centroid_z_px"] - best_z) < 1.5
                df_slice = df[z_mask]

                if not df_slice.empty:
                    # Scale coordinates to match the display grid
                    scale_y = disp_raw.shape[0] / max(1, raw_slice.shape[0])
                    scale_x = disp_raw.shape[1] / max(1, raw_slice.shape[1])

                    pts_x = df_slice["centroid_x_px"].values * scale_x
                    pts_y = df_slice["centroid_y_px"].values * scale_y

                    # Plot distinct red dots with white borders for high visibility
                    ax_overlay.scatter(pts_x, pts_y, s=15, c='red', alpha=0.8, edgecolors='white', linewidths=0.5)

        plot_slice_fast(axes[0, 0], axes[0, 1], img_full_seg, df_full, "FULL")
        plot_slice_fast(axes[1, 0], axes[1, 1], img_crop_seg, df_crop, "CROP")

        for ax in axes.ravel(): ax.axis("off")
        plt.tight_layout()
        plt.show()

    with out_tbl:
        display(widgets.HTML("<div class='ns-header'>Crop Features Sample</div>"))


run_btn.on_click(on_run)

# ----------------------------
# Final UI Layout
# ----------------------------
inner2 = widgets.VBox([
    source_card,
    mask_input_card,
    segmentation_workflow,
    run_btn,
    widgets.Box([status_html], layout=widgets.Layout(width="100%", margin="10px 0")),
    out_log,
    out_fig,
    out_tbl
])
inner2.add_class("ns-root")

ui = widgets.VBox([
    style_html,
    inner2
], layout=widgets.Layout(width="90%", margin="0 auto"))

display(ui)

# Step 2: NucleiSky matching (3D)
---

### **Option A: Run NucleiSky3D Matching**

This is the core step where the “constellation” of nuclei in your crop is located within the reference volume.

**1. Matching Algorithms**

* **Pyramid (Tetrahedra):** Robust for sparse-to-medium constellations using scale-invariant tetrahedron descriptors.
* **Hashing3D:** Often best for dense constellations using invariant voting.

**2. Configuration**

* Use the default parameters first.
* If matching fails, adjust:
  * `scale_min / scale_max` (expected magnification ratio)
  * `inlier_radius_um` (nuclei localization + segmentation noise)
  * segmentation parameters (most common root cause)

**3. Outputs**

* `best_scale`, `best_R (3x3)`, `best_t (z,y,x in µm)`
* QC plots (MIP overlays)
* Optional exports: aligned crop TIFF + saved transform JSON


In [ ]:
#@title Run NucleiSky3D
# ============================================

# ----------------------------
# UI Styling
# ----------------------------
style_html = widgets.HTML(STYLE_CSS)


# ----------------------------
# Shared layout helpers
# ----------------------------
row_width = "300px"
description_width = "145px"
input_width = "130px"
checkbox_width = "24px"
tab_label_width = "220px"

def _field_row(label, widget, *, width="100%"):
    return widgets.HBox(
        [
            widgets.HTML(
                (
                    "<div style='font-size:13px; color:#334155; "
                    "white-space:nowrap; text-align:right; padding-right:8px;'>"
                    f"{html.escape(label)}</div>"
                ),
                layout=widgets.Layout(
                    width=description_width,
                    min_width=description_width,
                ),
            ),
            widget,
        ],
        layout=widgets.Layout(
            width=width,
            align_items="center",
            justify_content="flex-start",
        ),
    )

def _make_labeled_checkbox(label, *, value=False, width="100%"):
    widget = widgets.Checkbox(
        value=value,
        description="",
        indent=False,
        layout=widgets.Layout(width=checkbox_width, margin="0"),
    )
    return widget, _field_row(label, widget, width=width)

# ----------------------------
# Config Widget Builders
# ----------------------------
SAFE_RANGES = {
    "min_inliers_abs": (1, 10000),
    "min_inliers_frac": (0.0, 1.0),
    "min_inliers_hard_floor": (1, 1000),
    "min_inliers_cap_frac": (0.1, 1.0),
}

def _make_value_widget(key: str, val):
    layout = widgets.Layout(width=input_width)
    if isinstance(val, bool):
        return widgets.Checkbox(value=bool(val), description="", indent=False, layout=widgets.Layout(width=checkbox_width, margin="0"))
    if val is None:
        return widgets.Text(value="", placeholder="(leave blank for None)", layout=layout)

    if isinstance(val, int) and not isinstance(val, bool):
        if key in SAFE_RANGES:
            lo, hi = SAFE_RANGES[key]
            return widgets.BoundedIntText(value=int(val), min=int(lo), max=int(hi), layout=layout)
        return widgets.IntText(value=int(val), layout=layout)

    if isinstance(val, float):
        if key in SAFE_RANGES:
            lo, hi = SAFE_RANGES[key]
            return widgets.BoundedFloatText(value=float(val), min=float(lo), max=float(hi), layout=layout)
        return widgets.FloatText(value=float(val), layout=layout)

    return widgets.Text(value=str(val), layout=layout)

def _parse_text_value(text_val):
    s = str(text_val).strip()
    if not s or s.lower() == "none":
        return None
    if s.lower() == "true":
        return True
    if s.lower() == "false":
        return False
    try:
        f = float(s)
        return int(f) if f.is_integer() else f
    except ValueError:
        return s

def create_matcher_config_widget(cfg_init: dict):
    """Build 3D matcher configuration tabs with get hooks."""
    cfg_init = copy.deepcopy(cfg_init)
    sections = ["_common", "pyramid", "hashing3d"]

    section_meta = {
        "_common": (
            "General",
            "Global thresholds for all matchers (units in µm where applicable).",
        ),
        "pyramid": (
            "Pyramid (Tetra)",
            "RANSAC on tetrahedron descriptors; scale and rotation invariant.",
        ),
        "hashing3d": (
            "Hashing3D",
            "Geometric hashing on invariant coordinate frames.",
        ),
    }

    widget_map = {}
    tabs = []
    titles = []

    for sec in sections:
        if sec not in cfg_init:
            continue

        widget_map[sec] = {}
        rows = []

        title, desc = section_meta.get(sec, (sec, ""))
        titles.append(title)

        rows.append(widgets.HTML(f"<div class='ns-desc'>{html.escape(desc)}</div>"))

        items = cfg_init.get(sec, {})

        for k in sorted(items.keys()):
            w = _make_value_widget(k, items[k])
            widget_map[sec][k] = w

            label = widgets.HTML(
                (
                    "<div style='font-size:13px; color:#334155; "
                    "white-space:nowrap; text-align:right; padding-right:8px;'>"
                    f"{html.escape(str(k))}</div>"
                ),
                layout=widgets.Layout(width=tab_label_width, min_width=tab_label_width),
            )

            row = widgets.HBox(
                [label, w],
                layout=widgets.Layout(
                    width="100%",
                    margin="0",
                    align_items="center",
                    justify_content="flex-start",
                ),
            )
            row.add_class("ns-tab-row")
            rows.append(row)

        tab_body = widgets.VBox(
            rows,
            layout=widgets.Layout(padding="0", width="100%"),
        )
        tab_body.add_class("ns-tab-panel")
        tabs.append(tab_body)

    ui_tabs = widgets.Tab(children=tabs)
    ui_tabs.add_class("ns-tabs")

    for i, title in enumerate(titles):
        ui_tabs.set_title(i, title)

    def get_cfg():
        cfg = {sec: {} for sec in sections if sec in widget_map}

        for sec in cfg:
            for k, w in widget_map[sec].items():
                if isinstance(w, widgets.Text):
                    cfg[sec][k] = _parse_text_value(w.value)
                else:
                    cfg[sec][k] = w.value

        return cfg

    return ui_tabs, get_cfg, lambda cfg: None


# ----------------------------
# Small helpers (library-first)
# ----------------------------
def _margin_um_to_px_zyx(margin_um: float, voxel_full_um_zyx) -> tuple[int, int, int]:
    """Convert a (scalar) margin in µm into integer pixels for (Z,Y,X)."""
    vox = np.asarray(voxel_full_um_zyx, dtype=float).reshape(3,)
    margin_um = float(margin_um)
    return tuple(int(np.ceil(margin_um / max(v, 1e-12))) for v in vox)


# ----------------------------
# UI Definitions
# ----------------------------
info_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Pipeline Overview</div>"),
    widgets.HTML(
        "<div class='ns-body-text'>"
        "This step executes <b>two 3D pattern matching algorithms</b> to find the crop location:"
        "<ul style='margin-top:4px; margin-bottom:4px;'>"
        "<li><b>Pyramid (Tetra):</b> Highly robust scale/rotation invariant tetrahedron matching.</li>"
        "<li><b>Hashing3D:</b> Voting-based geometric alignment for massive or dense point clouds.</li>"
        "</ul>"
        "</div>"
    )
]).add_class("ns-card")

MATCHER_UI_TABS, get_cfg, _ = create_matcher_config_widget(DEFAULT_MATCHER_CONFIG)
use_default_box, row_use_default_box = _make_labeled_checkbox("Use default settings", value=True, width="260px")

customize_box = widgets.VBox([widgets.HTML("<div style='height:8px'></div>"), MATCHER_UI_TABS])
customize_box.layout.display = "none"
use_default_box.observe(
    lambda _: setattr(customize_box.layout, "display", "none" if use_default_box.value else "block"),
    names="value",
)

config_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Matcher Parameters</div>"),
    row_use_default_box,
    customize_box,
]).add_class("ns-card")

export_region = widgets.Dropdown(
    options=[("ROI (Bounding Box)", "roi"), ("Full volume (Padded)", "full")],
    value="roi",
    layout=widgets.Layout(width="170px"),
)
row_export_region = _field_row("Export region", export_region)

export_margin_um = widgets.FloatText(
    value=10.0,
    layout=widgets.Layout(width=input_width),
)
row_export_margin_um = _field_row("ROI margin (µm)", export_margin_um)

export_enabled, row_export_enabled = _make_labeled_checkbox("Export aligned TIFFs", value=True)
export_uint16, row_export_uint16 = _make_labeled_checkbox("Convert to uint16", value=False)

save_qc, row_save_qc = _make_labeled_checkbox("Save QC overlays", value=True)
show_qc, row_show_qc = _make_labeled_checkbox("Show QC inline", value=True)

export_grid = widgets.GridBox(
    children=[
        row_export_region,
        row_export_margin_um,
        row_export_enabled,
        row_export_uint16,
        row_save_qc,
        row_show_qc,
    ],
    layout=widgets.Layout(
        width="100%",
        grid_template_columns="repeat(2, minmax(340px, 1fr))",
        grid_gap="10px 24px",
        align_items="center",
    ),
)

export_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Export Options</div>"),
    export_grid,
]).add_class("ns-card")

run_btn = widgets.Button(
    description="Run NucleiSky3D",
    button_style="primary",
    icon="bolt",
    layout=widgets.Layout(width="100%", height="45px", font_weight="bold"),
)
run_btn.add_class("ns-primary-button")
ui_out = widgets.Output(
    layout=widgets.Layout(
        max_height="600px",
        overflow="auto",
        border="1px solid var(--ns-border)",
        padding="8px",
        background_color="var(--ns-bg-soft)",
    )
)
status_html = widgets.HTML("", layout=widgets.Layout(width="100%"))


# ----------------------------
# Main runner (library-first I/O + consistent grid)
# ----------------------------
def _run_matching_pipeline(cfg_selected: dict):
    # Ensure required state exists
    for key in ("labels_full_seg", "labels_crop_seg", "voxel_full_seg", "voxel_crop_seg"):
        if key not in globals():
            raise RuntimeError(f"Missing '{key}'. Please run Step 2 first.")
    for key in ("img_full", "img_crop", "voxel_size_full_um", "voxel_size_crop_um"):
        if key not in globals():
            raise RuntimeError(f"Missing '{key}'. Please run Step 1 first.")
    if "RESULT_DIR" not in globals():
        raise RuntimeError("Missing 'RESULT_DIR'. Please run the setup cell / Step 0.")

    # Matching inputs: centroids (always in µm). Use seg voxel sizes only to convert px->µm if needed.
    voxel_full_seg = tuple(float(v) for v in globals()["voxel_full_seg"])
    voxel_crop_seg = tuple(float(v) for v in globals()["voxel_crop_seg"])
    centroids_full_um = centroids_from_df_3d(df_full, voxel_size_um_zyx=np.array(voxel_full_seg), name="df_full")
    centroids_crop_um = centroids_from_df_3d(df_crop, voxel_size_um_zyx=np.array(voxel_crop_seg), name="df_crop")

    # Export inputs: raw/high-res grids
    img_full_raw = np.asarray(globals()["img_full"])
    img_crop_raw = np.asarray(globals()["img_crop"])
    voxel_full_raw = tuple(float(v) for v in globals()["voxel_size_full_um"])
    voxel_crop_raw = tuple(float(v) for v in globals()["voxel_size_crop_um"])
    full_shape_raw = tuple(int(v) for v in img_full_raw.shape[:3])
    crop_shape_raw = tuple(int(v) for v in img_crop_raw.shape[:3])

    labels_full_seg = np.asarray(globals()["labels_full_seg"])
    labels_crop_seg = np.asarray(globals()["labels_crop_seg"])
    print(f"📐 Segmented Grid Context: Full={labels_full_seg.shape}, Crop={labels_crop_seg.shape}")
    print(f"📐 Original Grid Context:  Full={img_full_raw.shape}, Crop={img_crop_raw.shape}")

    # Output directory
    out_dir = Path(globals()["RESULT_DIR"]) / "matching" / "exports_nucleisky3d"
    out_dir.mkdir(parents=True, exist_ok=True)
    save_matcher_config_json(cfg_selected, out_dir / "matcher_config.json")

    results_map: dict[str, dict] = {}
    records_map: dict[str, dict] = {}

    for matcher_name in ("pyramid", "hashing3d"):
        print(f"\n⚙️ Running {matcher_name} matcher...")
        try:
            # Run matching directly in the ORIGINAL (raw) grid to avoid any bbox rescaling later.
            res = NucleiSky3D(
                centroids_crop_um=centroids_crop_um,
                centroids_full_um=centroids_full_um,
                full_shape_px_zyx=full_shape_raw,
                crop_shape_px_zyx=crop_shape_raw,
                pixel_size_full_um_zyx=voxel_full_raw,
                pixel_size_crop_um_zyx=voxel_crop_raw,
                matcher=matcher_name,
                matcher_config=cfg_selected,
                df_full=df_full,
                df_crop=df_crop,
            )
            results_map[matcher_name] = res

            mq = res.get("match_quality") or {}
            print(f"  success={res.get('success')}  frac_inliers={mq.get('frac_inliers')}  mean_error_um={mq.get('mean_error_um')}")

            # Ensure bbox is present + consistent with raw export grid (library geometry helper).
            if res.get("success") and res.get("best_scale") is not None:
                res = dict(res)
                res["best_bbox"] = tuple(
                    int(v) for v in bbox_full_px_from_similarity_um_3d(
                        crop_shape_px=crop_shape_raw,
                        pixel_size_full_um_zyx=voxel_full_raw,
                        pixel_size_crop_um_zyx=voxel_crop_raw,
                        scale=res["best_scale"],
                        R_zyx=res["best_R"],
                        t_um_zyx=res["best_t"],
                        margin_um=0.0,
                        full_shape_px=full_shape_raw,
                    )
                )
                results_map[matcher_name] = res

            # ✅ QC: ONE FIGURE PER MATCHER using plot_warp_overlay3D (FULL + ROI + projections)
            if res.get("best_scale") is not None and (save_qc.value or show_qc.value):
                qc_path = out_dir / f"{matcher_name}_qc_overlay3d.png" if save_qc.value else None
                plot_warp_overlay3D(
                    img_full_zyx=img_full_raw,
                    img_crop_zyx=img_crop_raw,
                    record_or_result=res,
                    pixel_size_full_um_zyx=voxel_full_raw,
                    pixel_size_crop_um_zyx=voxel_crop_raw,
                    roi_margin_um=float(export_margin_um.value) if export_region.value == "roi" else 0.0,
                    max_display_dim=512,
                    save_path=qc_path,
                    show=bool(show_qc.value),
                    title=f"{matcher_name} QC (FULL + ROI slice + MIP)",
                )
                if save_qc.value:
                    print(f"  🖼️ QC saved: {qc_path.name}")

            # Persist transform record using library I/O (JSON-safe + consistent schema)
            if res.get("success") and res.get("best_scale") is not None:
                rec = save_nucleisky_transform_3d(
                    res,
                    out_path=out_dir / f"{matcher_name}_transform.json",
                    matcher_name=matcher_name,
                    pixel_size_full_um_zyx=voxel_full_raw,
                    pixel_size_crop_um_zyx=voxel_crop_raw,
                    require_success=True,
                )
                append_transform_jsonl(rec, out_dir / "transforms.jsonl")
                records_map[matcher_name] = rec

        except Exception:
            print(f"❌ {matcher_name} crashed.")
            traceback.print_exc()
            results_map[matcher_name] = dict(success=False, matcher=matcher_name)

    # Exports
    if export_enabled.value:
        print("\n💾 Exporting successful results...")

    for matcher_name, rec in records_map.items():
        if not isinstance(rec, dict) or not rec.get("success"):
            continue
        if not export_enabled.value:
            continue

        if export_region.value == "roi" and rec.get("bbox_full_px_z0z1y0y1x0x1") is not None:
            roi_dir = out_dir / f"{matcher_name}_images_roi"
            margin_px_zyx = _margin_um_to_px_zyx(float(export_margin_um.value), voxel_full_raw)

            export_bbox_pair_tiffs_3d(
                img_full_zyx=img_full_raw,
                img_crop_zyx=img_crop_raw,
                record_or_result=rec,
                voxel_full_um_zyx=voxel_full_raw,
                voxel_crop_um_zyx=voxel_crop_raw,
                out_dir=roi_dir,
                margin_px_zyx=margin_px_zyx,
                prefix=f"{matcher_name}_",
            )

            print(f"  ✅ {matcher_name}: ROI export -> {roi_dir.name}/")

        else:
            out_path = out_dir / f"aligned_crop_{matcher_name}.tif"
            export_aligned_crop_tiff(
                img_full=img_full_raw,
                img_crop=img_crop_raw,
                output_path=out_path,
                pixel_size_full_um=voxel_full_raw,
                pixel_size_crop_um=voxel_crop_raw,
                best_scale=float(rec["scale"]),
                best_R=np.asarray(rec["R_zyx"], float),
                best_t=np.asarray(rec["t_um_zyx"], float),
                as_uint16_if_float=bool(export_uint16.value),
                export_region="full",
            )
            print(f"  ✅ {matcher_name}: Full grid export -> {out_path.name}")

    # Pick best using library scorer (based on saved-record schema)
    best_record = None
    shortlist = []
    try:
        if records_map:
            best_record, shortlist = pick_best_transform_3d(list(records_map.values()))
    except Exception:
        best_record, shortlist = None, []

    globals()["res_pyramid"] = results_map.get("pyramid")
    globals()["res_hashing"] = results_map.get("hashing3d")
    globals()["best_record_3d"] = best_record
    globals()["transform_records_3d"] = records_map

    return out_dir, best_record

def _on_run_clicked(_):
    run_btn.disabled = True
    status_html.value = _status_info("Running the 3D matching pipeline.", title="Processing")
    ui_out.clear_output(wait=True)

    with ui_out:
        try:
            out_dir, best_rec = _run_matching_pipeline(DEFAULT_MATCHER_CONFIG if use_default_box.value else get_cfg())
            lines = [f"✅ Results saved under: {out_dir.name}/"]
            if not best_rec:
                lines.append("⚠️ No confident match was found.")
                status_html.value = _status_err(lines, title="Matching finished with warnings")
            else:
                lines.append(f"🏆 Best matcher: {best_rec.get('matcher')}")
                status_html.value = _status_ok(lines)
        except Exception as e:
            status_html.value = _status_err([str(e)], title="Matching failed")
            traceback.print_exc()
        finally:
            run_btn.disabled = False

run_btn.on_click(_on_run_clicked)

title_html = widgets.HTML(
    "<div class='ns-page-title'>Run NucleiSky3D</div>"
    "<div class='ns-page-subtitle'>Run pyramid and hashing-based 3D matching, then export the selected results.</div>"
)

inner3 = widgets.VBox([
    title_html,
    info_card,
    config_card,
    export_card,
    run_btn,
    widgets.Box([status_html], layout=widgets.Layout(width="100%", margin="10px 0")),
    ui_out
])
inner3.add_class("ns-root")

ui = widgets.VBox([
    style_html,
    inner3
], layout=widgets.Layout(width="90%", margin="0 auto"))

display(ui)


### **Option B: Run Adaptive Matching (3D)**

This “autopilot” mode automatically attempts multiple matching strategies (Pyramid and Hashing3D) in a sequence designed to work well across different nuclei densities. It stops as soon as a high-confidence match is found, and can optionally export an aligned crop TIFF.

* **Random seed:** Makes results reproducible when random sampling is used.
* **Scale bounds:** Restricts search to plausible magnification ratios.
* **Outcome:** The best valid transform is automatically applied, visualized, and exported into the `adaptive/` results folder.


In [ ]:
#@title Run Adaptive NucleiSky3D
# ============================================

# ----------------------------
# UI Styling
# ----------------------------
style_html = widgets.HTML(STYLE_CSS)

# ----------------------------
# UI Components
# ----------------------------

# 1) Header Card
header_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Adaptive 3D Matching Strategy</div>"),
    widgets.HTML(
        "<div class='ns-body-text'>"
        "This step runs a <b>sequential cascade</b> of 3D matchers (pyramid + hashing3d). "
        "It tries algorithms one by one until a valid 3D transform is found, then exports the best result. "
        "QC is generated as a <b>single consolidated figure</b> per run using <code>plot_warp_overlay3D</code>."
        "</div>"
    ),
])
header_card.add_class("ns-card")

layout_width = "520px"
description_width = "300px"
input_width = "200px"

def _field_row(label, widget, *, width=layout_width):
    return widgets.HBox(
        [
            widgets.HTML(
                f"<div style='font-size:13px; color:#334155; white-space:nowrap; text-align:right; padding-right:8px;'>{html.escape(label)}</div>",
                layout=widgets.Layout(width=description_width, min_width=description_width),
            ),
            widget,
        ],
        layout=widgets.Layout(
            width=width,
            align_items="center",
            justify_content="flex-start",
        ),
    )


def _make_value_widget(widget_cls, label, *, width=input_width, **kwargs):
    widget = widget_cls(layout=widgets.Layout(width=width), **kwargs)
    return widget, _field_row(label, widget)

def _make_checkbox(label, *, value=False):
    widget = widgets.Checkbox(
        value=value,
        description="",
        indent=False,
        layout=widgets.Layout(width="24px", margin="0"),
    )
    return widget, _field_row(label, widget)

# 2) Settings Card
w_base_seed, row_base_seed = _make_value_widget(widgets.IntText, "Base Seed", value=42)

w_inlier_radius, row_inlier_radius = _make_value_widget(widgets.FloatText, "inlier_radius_um", value=2.0)
w_frac_inliers, row_frac_inliers = _make_value_widget(widgets.FloatText, "frac_inliers_thresh", value=0.6)
w_scale_min, row_scale_min = _make_value_widget(widgets.FloatText, "scale_min", value=0.5)
w_scale_max, row_scale_max = _make_value_widget(widgets.FloatText, "scale_max", value=2.0)

w_use_cfg_from_step3, row_use_cfg_from_step3 = _make_checkbox(
    "Start from Step 3 config (if available)", value=True
)

w_roi_margin_um, row_roi_margin_um = _make_value_widget(widgets.FloatText, "ROI margin (µm)", value=10.0)
w_show_qc, row_show_qc = _make_checkbox("Show QC inline", value=False)
w_save_qc, row_save_qc = _make_checkbox("Save QC PNG", value=True)

w_time_limit_chk, row_time_limit_chk = _make_checkbox("Enable time limit", value=False)
w_time_limit_val, row_time_limit_val = _make_value_widget(
    widgets.IntText, "Max seconds", value=60, disabled=True
)

def _toggle_time(_):
    w_time_limit_val.disabled = not w_time_limit_chk.value

w_time_limit_chk.observe(_toggle_time, names="value")

w_result_dir_root = widgets.Text(
    value="",
    layout=widgets.Layout(width="100%"),
)
row_result_dir_root = _field_row("RESULT_DIR root (optional)", w_result_dir_root, width="98%")

adv_ui = widgets.VBox([
    widgets.HTML("<div class='ns-desc'>Advanced execution constraints.</div>"),
    widgets.HBox([row_time_limit_chk, row_time_limit_val]),
])

adv_ui.add_class("ns-accordion-body")
adv_accord = widgets.Accordion(children=[adv_ui], titles=("Advanced Constraints",))
adv_accord.add_class("ns-accordion")

settings_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Configuration</div>"),
    widgets.HBox([row_base_seed, row_use_cfg_from_step3]),
    widgets.HTML("<div style='height:6px'></div>"),
    widgets.HBox([row_inlier_radius, row_frac_inliers, row_scale_min, row_scale_max]),
    widgets.HTML("<div style='height:6px'></div>"),
    widgets.HBox([row_roi_margin_um, row_show_qc, row_save_qc]),
    widgets.HTML("<div style='height:8px'></div>"),
    adv_accord,
    widgets.HTML("<div style='height:8px'></div>"),
    row_result_dir_root,
])
settings_card.add_class("ns-card")

# 3) Execution controls + outputs
run_btn = widgets.Button(
    description="Run Adaptive NucleiSky3D",
    button_style="primary",
    icon="rocket",
    layout=widgets.Layout(width="100%", height="45px", font_weight="bold"),
)
run_btn.add_class("ns-primary-button")

status_log = widgets.HTML(value="", layout=widgets.Layout(width="100%"))
out = widgets.Output(layout=widgets.Layout(max_height="600px", overflow="auto", border="1px solid var(--ns-border)", padding="8px"))


# ----------------------------
# Logic
# ----------------------------
def _preflight():
    required = (
        "df_full", "df_crop",
        "img_full", "img_crop",
        "voxel_size_full_um", "voxel_size_crop_um",
        "RESULT_DIR",
    )
    missing = [k for k in required if k not in globals()]
    if missing:
        raise RuntimeError(f"Missing globals: {missing}. Run Steps 1–2 first.")

    # Optional "seg" resources
    opt = ("img_full_seg", "img_crop_seg", "voxel_full_seg", "voxel_crop_seg", "labels_full_seg", "labels_crop_seg")
    return {k: globals().get(k) for k in opt}

def _resolve_result_dir_root() -> str:
    s = str(w_result_dir_root.value).strip()
    return str(Path(s)) if s else str(Path(globals()["RESULT_DIR"]))

def _resolve_cfg_selected() -> dict:
    # Start from Step 3 config if you stored it there, then override the core knobs.
    cfg_selected = None
    if w_use_cfg_from_step3.value and "_NS3D_MATCHER_CFG_SELECTED" in globals():
        cfg_selected = globals()["_NS3D_MATCHER_CFG_SELECTED"]

    overrides_common = dict(
        inlier_radius_um=float(w_inlier_radius.value),
        frac_inliers_thresh=float(w_frac_inliers.value),
        scale_min=float(w_scale_min.value),
        scale_max=float(w_scale_max.value),
        # NOTE: adaptive seed is controlled by base_seed in the orchestrator;
        # do NOT force random_state here unless your matchers use it directly.
    )

    if cfg_selected is None:
        return {"_common": overrides_common}

    cfg_selected = dict(cfg_selected)
    cfg_selected["_common"] = {**dict(cfg_selected.get("_common", {})), **overrides_common}
    return cfg_selected

def _time_limit_s() -> float | None:
    return float(w_time_limit_val.value) if w_time_limit_chk.value else None


def on_run_adaptive_3d(_):
    run_btn.disabled = True
    status_log.value = _status_info("Initializing adaptive 3D search.", title="Processing")
    out.clear_output()

    try:
        opt = _preflight()
        rdir = _resolve_result_dir_root()
        cfg_selected = _resolve_cfg_selected()
        max_time = _time_limit_s()
        base_seed = int(w_base_seed.value)
        roi_margin_um = float(w_roi_margin_um.value)

        # run_adaptive_matching_and_export_3d uses result_dir/matching/adaptive_3d/exports_adaptive/
        exports_dir = Path(rdir) / "matching" / "adaptive_3d" / "exports_adaptive"
        exports_dir.mkdir(parents=True, exist_ok=True)

        with out:
            print(f"📁 RESULT_DIR root: {rdir}")
            print(f"📁 Outputs will be under: {exports_dir}")
            print(f"🧪 Config overrides: inlier_radius={w_inlier_radius.value}, frac_inliers={w_frac_inliers.value}, "
                  f"scale=[{w_scale_min.value}, {w_scale_max.value}]")
            if max_time is not None:
                print(f"⏱️ Time limit: {max_time:.0f}s")

            best_out, history = run_adaptive_matching_and_export_3d(
                df_full=globals()["df_full"],
                df_crop=globals()["df_crop"],
                img_full_orig=globals()["img_full"],
                img_crop_orig=globals()["img_crop"],
                pixel_size_full_orig_um_zyx=globals()["voxel_size_full_um"],
                pixel_size_crop_orig_um_zyx=globals()["voxel_size_crop_um"],
                result_dir=str(Path(rdir)),
                cfg_selected=cfg_selected,
                base_seed=base_seed,
                store_full_out=False,
                max_total_time_s=max_time,
                img_full_seg=opt.get("img_full_seg"),
                img_crop_seg=opt.get("img_crop_seg"),
                pixel_size_full_seg_um_zyx=opt.get("voxel_full_seg"),
                pixel_size_crop_seg_um_zyx=opt.get("voxel_crop_seg"),
                labels_full=opt.get("labels_full_seg"),
                labels_crop=opt.get("labels_crop_seg"),
                save_segmentation_masks=True,
            )

            mq = best_out.get("match_quality", {}) if isinstance(best_out, dict) else {}
            print("\n✅ Adaptive finished.")
            print(f"   best matcher: {best_out.get('matcher')}  success={best_out.get('success')}")
            print(f"   frac_inliers={mq.get('frac_inliers')}  mean_error_um={mq.get('mean_error_um')}")

            # --- Visualization (single consolidated figure, 2D-parity style) ---
            if best_out and best_out.get("best_scale") is not None:
                print("\n🎨 Generating 3D QC overlay figure...")
                qc_path = exports_dir / "qc_overlay3d.png" if w_save_qc.value else None
                plot_warp_overlay3D(
                    img_full_zyx=np.asarray(globals()["img_full"]),
                    img_crop_zyx=np.asarray(globals()["img_crop"]),
                    record_or_result=best_out,  # accepts NucleiSky3D output
                    pixel_size_full_um_zyx=globals()["voxel_size_full_um"],
                    pixel_size_crop_um_zyx=globals()["voxel_size_crop_um"],
                    roi_margin_um=roi_margin_um,
                    save_path=qc_path,
                    show=bool(w_show_qc.value),
                    title=f"Adaptive 3D QC (matcher={best_out.get('matcher')}, roi_margin_um={roi_margin_um:g})",
                )
                if w_save_qc.value:
                    print(f"  🖼️ QC saved: {qc_path.name}")

        # UI summary
        if best_out and best_out.get("success"):
            score = (best_out.get("match_quality", {}) or {}).get("frac_inliers", 0.0)
            status_log.value = _status_ok([
                f"✅ Success! Adaptive 3D search complete.",
                f"Best matcher: {best_out.get('matcher')}",
                f"Best inlier score: {float(score):.2f}",
                f"Results saved to: {exports_dir}",
            ])
        else:
            status_log.value = _status_warn([
                "Search completed, but no confident match was found.",
                "Try increasing the time limit, relaxing thresholds, or checking segmentation quality.",
                f"See: {exports_dir / 'history.jsonl'}",
            ], title="No match")

    except Exception as e:
        status_log.value = _status_err([str(e)], title="Adaptive crashed")
        with out:
            import traceback
            traceback.print_exc()
    finally:
        run_btn.disabled = False


run_btn.on_click(on_run_adaptive_3d)


# ----------------------------
# Layout
# ----------------------------
inner4 = widgets.VBox([
    header_card,
    settings_card,
    run_btn,
    widgets.Box([status_log], layout=widgets.Layout(width="100%", margin="10px 0")),
    out,
])
inner4.add_class("ns-root")

ui = widgets.VBox([
    style_html,
    inner4,
], layout=widgets.Layout(width="90%", margin="0 auto"))

display(ui)

# Step 3: Apply saved NucleiSky3D transform
---

Once you have successfully matched a crop (e.g., using the nuclei channel), you can reuse that transformation to align other channels or related volumes **without re-running the matcher**.

**Inputs**

* **Transform JSON:** A JSON file containing `best_scale`, `best_R`, `best_t` (and voxel sizes).
* **Reference Volume (Full):** The target ZYX volume you want to align to.
* **Moving Volume (Crop):** The volume you want to warp into the reference coordinate system.

**Export**

This notebook exports an **aligned crop** as a full-size Z-stack TIFF in the reference coordinate system (mostly zeros outside the crop support). QC MIP overlays are generated for visual validation.


In [ ]:
#@title Apply Saved Transform (3D)
# ============================================

# ----------------------------
# UI Styling
# ----------------------------
style_html = widgets.HTML(STYLE_CSS)

# ----------------------------
# Defaults
# ----------------------------
def _default_transform_path():
    if "RESULT_DIR" in globals():
        p = Path(globals()["RESULT_DIR"]) / "matching" / "exports_nucleisky3d" / "transforms.jsonl"
        if p.exists():
            return str(p)
    return ""

def _default_full_path():
    if "FULL_PATH_USED" in globals():
        return str(globals()["FULL_PATH_USED"])
    return ""

def _default_crop_path():
    if "CROP_PATH_USED" in globals():
        return str(globals()["CROP_PATH_USED"])
    return ""

# ----------------------------
# ROI helpers
# ----------------------------
def _margin_um_to_px_3d(margin_um, voxel_size_full_um_zyx):
    vz, vy, vx = map(float, voxel_size_full_um_zyx)
    return (
        int(np.ceil(float(margin_um) / max(vz, 1e-12))),
        int(np.ceil(float(margin_um) / max(vy, 1e-12))),
        int(np.ceil(float(margin_um) / max(vx, 1e-12))),
    )

def _save_full_roi_tiff(full_roi: np.ndarray, out_path: Path, voxel_size_full_um_zyx):
    save_tiff_zyx(out_path, full_roi, voxel_size_um_zyx=voxel_size_full_um_zyx)

# ----------------------------
# Widgets
# ----------------------------
transform_path = widgets.Text(value=_default_transform_path(), description="Transform:", layout=widgets.Layout(width="98%"))
load_btn = widgets.Button(description="Load transforms", button_style="info", icon="folder-open", layout=widgets.Layout(width="180px"))
load_btn.add_class("ns-action-button")

record_dd = widgets.Dropdown(options=[("— load a file first —", -1)], value=-1, description="Record:", layout=widgets.Layout(width="98%"))

full_path = widgets.Text(value=_default_full_path(), description="Full path:", layout=widgets.Layout(width="98%"))
crop_path = widgets.Text(value=_default_crop_path(), description="Crop path:", layout=widgets.Layout(width="98%"))

channel_axis_ref = widgets.Dropdown(
    options=[("Auto/None", None), ("0", 0), ("1", 1), ("2", 2), ("3", 3), ("-1", -1)],
    value=None,
    description="Full ch axis:",
    layout=widgets.Layout(width="220px"),
)
channel_axis_mov = widgets.Dropdown(
    options=[("Auto/None", None), ("0", 0), ("1", 1), ("2", 2), ("3", 3), ("-1", -1)],
    value=None,
    description="Crop ch axis:",
    layout=widgets.Layout(width="220px"),
)

out_dir = widgets.Text(
    value=str(Path(globals().get("RESULT_DIR", ".")) / "apply_transform"),
    description="Out dir:",
    layout=widgets.Layout(width="98%"),
)

export_region = widgets.Dropdown(
    options=[("ROI only (recommended)", "roi"), ("Full volume", "full")],
    value="roi",
    description="Export:",
    layout=widgets.Layout(width="260px"),
)
roi_margin_um = widgets.FloatText(value=5.0, description="ROI margin (µm):", layout=widgets.Layout(width="260px"))

as_uint16 = widgets.Checkbox(value=False, description="Convert float output to uint16", indent=False)
show_qc = widgets.Checkbox(value=True, description="Show QC MIPs", indent=False)

apply_btn = widgets.Button(description="Apply & Export", button_style="primary", icon="play", layout=widgets.Layout(width="100%", height="40px"))
apply_btn.add_class("ns-primary-button")

status = widgets.HTML("", layout=widgets.Layout(width="100%"))
out = widgets.Output(layout=widgets.Layout(max_height="420px", overflow="auto", border="1px solid var(--ns-border)", padding="8px"))

# ----------------------------
# State
# ----------------------------
TRANSFORMS_3D = []

def _summarize_rec(rec: dict) -> str:
    matcher = rec.get("matcher", "unknown")
    scale = rec.get("best_scale", rec.get("scale", None))
    mq = rec.get("match_quality") or {}
    frac = mq.get("frac_inliers", None)
    line = rec.get("_line", None)
    src = Path(rec.get("_source_path", "")).name if rec.get("_source_path") else ""
    parts = [f"{matcher}"]
    if scale is not None:
        try:
            parts.append(f"scale={float(scale):.4g}")
        except Exception:
            parts.append(f"scale={scale}")
    if frac is not None:
        try:
            parts.append(f"frac_inliers={float(frac):.3g}")
        except Exception:
            parts.append(f"frac_inliers={frac}")
    if src:
        parts.append(f"{src}:{line}")
    return " | ".join(parts)

def _load_transforms():
    global TRANSFORMS_3D
    p = transform_path.value.strip()
    if not p:
        raise ValueError("Please provide a transform .json or .jsonl path.")
    recs = load_transforms_any_3d(p)
    if not recs:
        raise ValueError("No records found in transform file.")
    TRANSFORMS_3D = recs

    opts = [(_summarize_rec(rec), i) for i, rec in enumerate(TRANSFORMS_3D)]
    record_dd.options = opts
    record_dd.value = 0

def _maybe_warn_voxel_sizes(rec: dict, full_p: str, crop_p: str):
    warns = []
    try:
        vs_full = require_voxel_size_um_zyx(full_p, allow_missing_z=True)
    except Exception:
        vs_full = None
    try:
        vs_crop = require_voxel_size_um_zyx(crop_p, allow_missing_z=True)
    except Exception:
        vs_crop = None

    rec_full = rec.get("pixel_size_full_um_zyx", rec.get("pixel_size_full_orig_um_zyx"))
    rec_crop = rec.get("pixel_size_crop_um_zyx", rec.get("pixel_size_crop_orig_um_zyx"))

    def _fmt(v):
        if v is None:
            return "None"
        return tuple(float(x) for x in v)

    if vs_full is not None and rec_full is not None:
        a = np.asarray(vs_full, float).reshape(3,)
        b = np.asarray(rec_full, float).reshape(3,)
        if not np.allclose(a, b, rtol=1e-3, atol=1e-6):
            warns.append(f"Full voxel size mismatch: TIFF={_fmt(vs_full)} vs transform={_fmt(rec_full)}")
    if vs_crop is not None and rec_crop is not None:
        a = np.asarray(vs_crop, float).reshape(3,)
        b = np.asarray(rec_crop, float).reshape(3,)
        if not np.allclose(a, b, rtol=1e-3, atol=1e-6):
            warns.append(f"Crop voxel size mismatch: TIFF={_fmt(vs_crop)} vs transform={_fmt(rec_crop)}")
    return warns

def on_load(_):
    status.value = ""
    with out:
        clear_output()
    try:
        _load_transforms()
        status.value = _status_ok([f"Loaded {len(TRANSFORMS_3D)} transform record(s)."])
    except Exception as e:
        status.value = _status_err([str(e)], title="Load failed")
        with out:
            traceback.print_exc()

def on_apply(_):
    status.value = ""
    with out:
        clear_output()

    try:
        if not TRANSFORMS_3D:
            raise RuntimeError("No transforms loaded. Click 'Load transforms' first.")
        idx = int(record_dd.value)
        if idx < 0 or idx >= len(TRANSFORMS_3D):
            raise ValueError("Invalid transform selection.")
        rec = TRANSFORMS_3D[idx]

        fp = full_path.value.strip()
        cp = crop_path.value.strip()
        if not fp or not Path(fp).exists():
            raise FileNotFoundError("Full path is missing or invalid.")
        if not cp or not Path(cp).exists():
            raise FileNotFoundError("Crop path is missing or invalid.")

        warns = _maybe_warn_voxel_sizes(rec, fp, cp)
        if warns:
            status.value = _status_warn(warns, title="Voxel size mismatch warning")

        outdir = Path(out_dir.value.strip() or ".")
        outdir.mkdir(parents=True, exist_ok=True)

        # Extract transform parameters from record
        matcher = rec.get("matcher", "unknown")
        scale = rec.get("best_scale", rec.get("scale"))
        R = rec.get("best_R", rec.get("R_zyx"))
        t = rec.get("best_t", rec.get("t_um_zyx"))
        vox_full = rec.get("pixel_size_full_um_zyx", rec.get("pixel_size_full_orig_um_zyx"))
        vox_crop = rec.get("pixel_size_crop_um_zyx", rec.get("pixel_size_crop_orig_um_zyx"))

        if export_region.value == "roi":
            bbox = rec.get("bbox_full_px_z0z1y0y1x0x1") or rec.get("bbox_full_px_z0z1y0y1x0x1".lower())
            if bbox is None:
                with out:
                    print("⚠️ ROI requested but record has no bbox; falling back to full export.")
                export_region.value = "full"

        if export_region.value == "full":
            with out:
                print("💾 Exporting full aligned crop...")
                ret = run_export_with_record_3d(
                    rec=rec,
                    ref_path=fp,
                    mov_path=cp,
                    out_dir=str(outdir),
                    channel_axis_ref=channel_axis_ref.value,
                    channel_axis_mov=channel_axis_mov.value,
                    as_uint16_if_float=bool(as_uint16.value),
                )
                print("Export result:", ret)
        else:
            # ROI export path (mirrors Step 3 ROI export)
            if scale is None or R is None or t is None:
                raise ValueError("Selected record is missing transform parameters (scale/R/t).")
            if vox_full is None or vox_crop is None:
                raise ValueError("Selected record is missing voxel sizes.")

            img_full = load_volume(fp, channel_axis=channel_axis_ref.value)
            img_crop = load_volume(cp, channel_axis=channel_axis_mov.value)

            bbox6 = tuple(int(v) for v in (rec.get("bbox_full_px_z0z1y0y1x0x1") or []))
            if len(bbox6) != 6:
                raise ValueError("bbox_full_px_z0z1y0y1x0x1 must be length-6 for ROI export.")

            bbox_m = bbox_add_margin_px_3d(
                bbox6,
                margin_px=_margin_um_to_px_3d(float(roi_margin_um.value), vox_full),
                shape_zyx=img_full.shape,
            )
            z0, z1, y0, y1, x0, x1 = bbox_m
            full_roi = np.asarray(img_full[z0:z1, y0:y1, x0:x1])

            origin_um = np.array([z0, y0, x0], float) * np.array(vox_full, float)
            t_roi = np.asarray(t, float).reshape(3,) - origin_um
            qc_rec_roi = {"best_scale": scale, "best_R": R, "best_t": t_roi}

            roi_dir = outdir / f"{matcher}_roi"
            roi_dir.mkdir(parents=True, exist_ok=True)

            full_roi_path = roi_dir / f"{matcher}_full_roi.tif"
            _save_full_roi_tiff(full_roi, full_roi_path, vox_full)

            aligned_roi_path = roi_dir / f"{matcher}_aligned_crop_roi.tif"
            export_aligned_crop_tiff(
                img_full=full_roi,
                img_crop=img_crop,
                output_path=aligned_roi_path,
                pixel_size_full_um=vox_full,
                pixel_size_crop_um=vox_crop,
                best_scale=scale,
                best_R=R,
                best_t=t_roi,
                as_uint16_if_float=bool(as_uint16.value),
            )

            with out:
                print("💾 ROI export complete:")
                print("  ", full_roi_path)
                print("  ", aligned_roi_path)

            if show_qc.value:
                with out:
                    print("\n🖼️ QC MIP overlays (ROI)...")
                plot_warp_overlay3d(
                    img_full_zyx=full_roi,
                    img_crop_zyx=img_crop,
                    record_or_result=qc_rec_roi,
                    pixel_size_full_um_zyx=vox_full,
                    pixel_size_crop_um_zyx=vox_crop,
                    include_roi=False,
                    save_path=None,
                    show=True,
                    title=f"{matcher} QC (ROI)",
                )

        # Optional QC MIPs for full export (already shown for ROI)
        if show_qc.value and export_region.value == "full":
            with out:
                print("\n🖼️ QC MIP overlays (full)...")
            img_full = load_volume(fp, channel_axis=channel_axis_ref.value)
            img_crop = load_volume(cp, channel_axis=channel_axis_mov.value)
            plot_warp_overlay3d(
                img_full_zyx=img_full,
                img_crop_zyx=img_crop,
                record_or_result=rec,
                pixel_size_full_um_zyx=vox_full,
                pixel_size_crop_um_zyx=vox_crop,
                save_path=None,
                show=True,
                title=f"{matcher} QC (full)",
            )

        status.value = status.value + _status_ok([f"Done. Outputs in: {outdir}"], title="Export complete")

    except Exception as e:
        status.value = _status_err([str(e)], title="Apply failed")
        with out:
            traceback.print_exc()

load_btn.on_click(on_load)
apply_btn.on_click(on_apply)

# ----------------------------
# Layout
# ----------------------------
file_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Load transform file</div>"),
    widgets.HTML("<div class='ns-desc'>Use a JSON (single transform) or JSONL (multiple runs/matchers).</div>"),
    transform_path,
    load_btn,
    record_dd,
])
file_card.add_class("ns-card")

inputs_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Input volumes</div>"),
    full_path,
    crop_path,
    widgets.HBox([channel_axis_ref, channel_axis_mov]),
])
inputs_card.add_class("ns-card")

out_card = widgets.VBox([
    widgets.HTML("<div class='ns-header'>Export</div>"),
    out_dir,
    widgets.HBox([export_region, roi_margin_um]),
    widgets.HBox([as_uint16, show_qc]),
    apply_btn,
])
out_card.add_class("ns-card")

title_html = widgets.HTML(
    "<div class='ns-page-title'>Apply Saved NucleiSky3D Transform</div>"
    "<div class='ns-page-subtitle'>Load a saved 3D transform and export the aligned crop volume.</div>"
)

inner5 = widgets.VBox([
    title_html,
    file_card,
    inputs_card,
    out_card,
    widgets.Box([status], layout=widgets.Layout(width="100%", margin="10px 0")),
    out,
])
inner5.add_class("ns-root")

ui = widgets.VBox([
    style_html,
    inner5,
], layout=widgets.Layout(width="90%", margin="0 auto"))

display(ui)
